# Notebook 87 — Paired HITL Development Draft, Blind Review, Freeze, and A1 Cleanup Truth Experiment

**Purpose:** take the existing paired annotated pages through the same YOLO draft/confidence-map and human review workflow used for production HITL, while keeping the paired-development population physically and logically separate from the real-world HITL population and keeping clean truth inaccessible until the reviewed-mask and observable-output locks are closed.

## Population separation

- Real-world HITL remains under `human_in_the_middle/` and is never written by this notebook.
- Paired annotated pages are read from `source/annotated/`.
- Clean pages remain under `source/clean/` and are not resolved until the truth-scoring stage.
- All draft, confidence, review, audit and lock artefacts are written under `paired_hitl_development/`.

## Predeclared cleanup candidates

All candidates start from **A1**, where V7.8 `AMBIGUOUS` pixels retain exact observed bytes. Additional cleanup is permitted only at A1-preserved AMBIGUOUS pixels:

- `A1_BASELINE`: no additional cleanup.
- `C090_P010`: `p(annotation) >= 0.900` and `p_print < 0.10` → paper.
- `C0875_P010`: `p(annotation) >= 0.875` and `p_print < 0.10` → paper.
- `C085_P010`: `p(annotation) >= 0.850` and `p_print < 0.10` → paper.

No threshold selection or promotion is performed automatically.

In [ ]:
# @title
# ============================================================
# 0. Mount Drive and install runtime dependencies
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

!pip install -q ultralytics gradio

Mounted at /content/drive


In [ ]:
# @title
# ============================================================
# 1. Configuration and imports
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from dataclasses import asdict
import hashlib, json, shutil, gc, random, re

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')

# Existing paired data. CLEAN_DIR is intentionally not used
# before the truth-scoring stage.
ANNOTATED_DIR = ROOT / 'source' / 'annotated'
CLEAN_DIR = ROOT / 'source' / 'clean'

# Completely separate experimental HITL namespace.
PH = ROOT / 'paired_hitl_development'
DRAFT_MASK_DIR = PH / 'draft_masks'
CONFIDENCE_DIR = PH / 'confidence_maps'
DRAFT_AUDIT_DIR = PH / 'draft_audit'
REVIEWED_MASK_DIR = PH / 'reviewed_masks'
REVIEW_AUDIT_DIR = PH / 'audit'
LOCK_DIR = PH / 'locks'
RUNS_DIR = PH / 'runs'

for folder in [PH, DRAFT_MASK_DIR, CONFIDENCE_DIR, DRAFT_AUDIT_DIR,
               REVIEWED_MASK_DIR, REVIEW_AUDIT_DIR, LOCK_DIR, RUNS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

MODEL_PATH = ROOT / 'training_runs' / '20261006_150912_yolo11n-seg' / 'weights' / 'best.pt'
V89_OPERATOR_PATH = ROOT / 'v8_9_release' / 'v8_9_operator.py'
P_PRINT_BUNDLE = ROOT / 'observed_print_full_development_freeze_v8_2f'/'20261117_120424_v8_2f_full_development_p_print_freeze'/'p_print_provider'
REMOVE_BUNDLE = ROOT / 'm5_remove_full_development_freeze_v8_8f'/'20261117_110512_notebook52f_v8_8_full_development_remove_freeze'/'full_development_remove'

# Override only if your final paths differ.
SOURCE_GROUP_MANIFEST_CANDIDATES = [
    ROOT / 'development_score_manifest.csv',
    ROOT / 'tables' / 'development_score_manifest.csv',
]

PREDICTION_CONFIDENCE = 0.20
TILE_SIZE = 640
TILE_STRIDE = 320
MODEL_IMAGE_SIZE = 640
MINIMUM_DRAFT_COMPONENT_AREA = 10
FORCE_REBUILD_DRAFTS = False

REVIEWER_CODE = 'DB'  # change before review
RANDOM_SEED = 42
SUPPORTED = {'.jpg','.jpeg','.png','.tif','.tiff','.bmp'}

CANDIDATES = {
    'A1_BASELINE': None,
    'C090_P010': (0.900, 0.10),
    'C0875_P010': (0.875, 0.10),
    'C085_P010': (0.850, 0.10),
}

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_notebook87')
RUN = RUNS_DIR / RUN_ID
for sub in ['tables','audit','pages','visuals','upload']:
    (RUN / sub).mkdir(parents=True, exist_ok=True)


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()


def md5_file(path, chunk=1024*1024):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(chunk), b''):
            h.update(block)
    return h.hexdigest()


def image_files(folder):
    return sorted(p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED)


def read_gray(path):
    a = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    require(a is not None, f'Unable to read image: {path}')
    return a

print('Run:', RUN)
print('CUDA:', torch.cuda.is_available())

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Run: /content/drive/MyDrive/MusicAnnotationProject/paired_hitl_development/runs/20261121_135130_notebook87
CUDA: False


## Stage A — Generate production-style drafts and confidence maps

This stage uses the same frozen YOLO checkpoint, threshold, tile size and overlap as the production HITL draft generator, but writes only to `paired_hitl_development/`.

In [ ]:
# @title
# ============================================================
# 2. Build paired annotated roster without touching clean truth
# ============================================================

annotated_map = {p.stem: p for p in image_files(ANNOTATED_DIR)}
require(annotated_map, f'No paired annotated pages found in {ANNOTATED_DIR}')

manifest_path = next((p for p in SOURCE_GROUP_MANIFEST_CANDIDATES if p.exists()), None)
manifest = pd.read_csv(manifest_path, dtype=str).fillna('') if manifest_path else pd.DataFrame()
manifest_by_page = manifest.set_index('page') if len(manifest) and 'page' in manifest.columns else None

roster_rows = []
for page, source in sorted(annotated_map.items()):
    source_group = ''
    if manifest_by_page is not None and page in manifest_by_page.index and 'source_group' in manifest_by_page.columns:
        source_group = str(manifest_by_page.loc[page, 'source_group'])
    roster_rows.append({
        'page': page,
        'page_key': f'paired_hitl_development__{page}',
        'population_role': 'PAIRED_HITL_DEVELOPMENT',
        'source_group': source_group,
        'annotated_path': str(source),
        'annotated_md5': md5_file(source),
    })

roster = pd.DataFrame(roster_rows)
roster.to_csv(RUN / 'tables' / 'paired_annotated_roster.csv', index=False)
print('Paired annotated pages:', len(roster))
display(roster.head())

Paired annotated pages: 83


,page,page_key,population_role,source_group,annotated_path,annotated_md5
0,Page002,paired_hitl_development__Page002,PAIRED_HITL_DEVELOPMENT,Wright_and_Round_Faith,/content/drive/MyDrive/MusicAnnotationProject/...,4c8af1a2739070167aa20b5669f81e7a
1,Page003,paired_hitl_development__Page003,PAIRED_HITL_DEVELOPMENT,Anglo_Music_Press_HotH,/content/drive/MyDrive/MusicAnnotationProject/...,2c913acf0379453343847ff904ecd337
2,Page004,paired_hitl_development__Page004,PAIRED_HITL_DEVELOPMENT,Anglo_Music_Press_HotH,/content/drive/MyDrive/MusicAnnotationProject/...,b085842548b126801c22a7137ed3b203
3,Page005,paired_hitl_development__Page005,PAIRED_HITL_DEVELOPMENT,Wright_and_Round_IwtBF,/content/drive/MyDrive/MusicAnnotationProject/...,adb2106f1d9c23f1a0b2c9ff1ea8660f
4,Page006,paired_hitl_development__Page006,PAIRED_HITL_DEVELOPMENT,Pennine_Les_Mis,/content/drive/MyDrive/MusicAnnotationProject/...,56b0ee07abebed4a8ee822118ccbdd5b


In [ ]:
# @title
# ============================================================
# 3. Frozen YOLO confidence-map generator
# ============================================================

require(MODEL_PATH.exists(), f'Model not found: {MODEL_PATH}')
#require(torch.cuda.is_available(), 'Select a T4 GPU runtime for draft generation.')

MODEL = YOLO(str(MODEL_PATH))
MODEL_MD5 = md5_file(MODEL_PATH)


def tile_positions(n):
    if n <= TILE_SIZE:
        return [0]
    p = list(range(0, n - TILE_SIZE + 1, TILE_STRIDE))
    if p[-1] != n - TILE_SIZE:
        p.append(n - TILE_SIZE)
    return p


def confidence_map(gray):
    colour = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    h, w = gray.shape
    ph, pw = max(h, TILE_SIZE), max(w, TILE_SIZE)
    canvas = np.full((ph, pw, 3), 255, np.uint8)
    canvas[:h, :w] = colour
    conf = np.zeros((ph, pw), np.float32)

    for y in tile_positions(ph):
        for x in tile_positions(pw):
            result = MODEL.predict(
                canvas[y:y+TILE_SIZE, x:x+TILE_SIZE],
                conf=PREDICTION_CONFIDENCE,
                imgsz=MODEL_IMAGE_SIZE,
                device=0,
                verbose=False,
                retina_masks=True,
            )[0]
            if result.masks is None:
                continue
            masks = result.masks.data.cpu().numpy()
            scores = result.boxes.conf.cpu().numpy()
            for mask, score in zip(masks, scores):
                mask = cv2.resize(mask, (TILE_SIZE, TILE_SIZE), interpolation=cv2.INTER_NEAREST) > 0.5
                region = conf[y:y+TILE_SIZE, x:x+TILE_SIZE]
                use = mask & (score > region)
                region[use] = score

    return conf[:h, :w]


def remove_small_components(mask, minimum_area=MINIMUM_DRAFT_COMPONENT_AREA):
    n, labels, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), 8)
    out = np.zeros(mask.shape, np.uint8)
    for label in range(1, n):
        if int(stats[label, cv2.CC_STAT_AREA]) >= minimum_area:
            out[labels == label] = 1
    return out

In [ ]:
# @title
# ============================================================
# 4. Generate/resume paired drafts and confidence maps
# ============================================================

generation_rows = []

for _, r in roster.iterrows():
    page = str(r.page)
    source = Path(r.annotated_path)
    source_md5 = str(r.annotated_md5)

    mask_path = DRAFT_MASK_DIR / f'{page}_draft.png'
    conf_path = CONFIDENCE_DIR / f'{page}_confidence.npy'
    audit_path = DRAFT_AUDIT_DIR / f'{page}.json'

    current = False
    if not FORCE_REBUILD_DRAFTS and mask_path.exists() and conf_path.exists() and audit_path.exists():
        try:
            old = json.loads(audit_path.read_text())
            current = (
                old.get('source_md5') == source_md5 and
                old.get('model_md5') == MODEL_MD5 and
                float(old.get('prediction_confidence')) == PREDICTION_CONFIDENCE
            )
        except Exception:
            current = False

    if current:
        status = 'REUSED'
    else:
        gray = read_gray(source)
        conf = confidence_map(gray)
        binary = conf >= PREDICTION_CONFIDENCE
        binary = remove_small_components(binary)
        np.save(conf_path, conf.astype(np.float16))
        require(cv2.imwrite(str(mask_path), binary.astype(np.uint8) * 255), f'{page}: write draft failed')
        audit = {
            'created_utc': datetime.now(timezone.utc).isoformat(),
            'population_role': 'PAIRED_HITL_DEVELOPMENT',
            'clean_truth_available_to_reviewer': False,
            'source_file': str(source),
            'source_md5': source_md5,
            'model_path': str(MODEL_PATH),
            'model_md5': MODEL_MD5,
            'prediction_confidence': PREDICTION_CONFIDENCE,
            'minimum_component_area': MINIMUM_DRAFT_COMPONENT_AREA,
            'draft_pixels': int(binary.sum()),
            'confidence_map': str(conf_path),
            'draft_mask': str(mask_path),
            'shape': list(gray.shape),
        }
        audit_path.write_text(json.dumps(audit, indent=2))
        status = 'GENERATED'

    generation_rows.append({
        'page': page,
        'status': status,
        'draft_mask': str(mask_path),
        'confidence_map': str(conf_path),
        'draft_audit': str(audit_path),
    })
    print(page, status)

pd.DataFrame(generation_rows).to_csv(RUN / 'tables' / 'draft_generation_summary.csv', index=False)

## Stage B — Blind paired HITL review

The review interface below reads only the annotated page, draft mask and continuous detector-confidence map. It never resolves `source/clean`.

### Controls

- **Brush add:** paint pixels into the authoritative reviewed mask.
- **Brush erase:** erase pixels from the authoritative reviewed mask.
- **Confidence threshold display:** changes only the visual overlay, not the saved reviewed mask.
- **Save & next:** writes the reviewed mask and audit record under `paired_hitl_development/`.

If the Gradio version in Colab is incompatible with the canvas editor, export the displayed page/mask and use the existing Notebook 07 reviewer pointed at `paired_hitl_development/`; the namespace and audit contract remain the same.

In [ ]:
# @title
# ============================================================
# 5. Build review queue and helper visualisations
# ============================================================

review_rows = []
for _, r in roster.iterrows():
    page = str(r.page)
    reviewed_path = REVIEWED_MASK_DIR / f'{page}_reviewed.png'
    review_audit = REVIEW_AUDIT_DIR / f'{page}.json'
    review_rows.append({
        **r.to_dict(),
        'draft_mask_path': str(DRAFT_MASK_DIR / f'{page}_draft.png'),
        'confidence_path': str(CONFIDENCE_DIR / f'{page}_confidence.npy'),
        'reviewed_mask_path': str(reviewed_path),
        'review_audit_path': str(review_audit),
        'reviewed': bool(reviewed_path.exists() and review_audit.exists()),
    })

review_queue = pd.DataFrame(review_rows)
review_queue.to_csv(RUN / 'tables' / 'review_queue.csv', index=False)
print('Already reviewed:', int(review_queue.reviewed.sum()), '/', len(review_queue))


def confidence_overlay(gray, conf, threshold=0.20, alpha=0.45):
    base = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    conf = np.asarray(conf, np.float32)
    visible = conf >= threshold
    colour = np.zeros_like(base)
    # Rainbow: blue -> cyan -> green -> yellow -> red.
    heat = cv2.applyColorMap(np.clip(conf * 255, 0, 255).astype(np.uint8), cv2.COLORMAP_TURBO)
    heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB)
    out = base.copy()
    out[visible] = np.clip((1-alpha)*base[visible] + alpha*heat[visible], 0, 255).astype(np.uint8)
    return out


def mask_overlay(gray, mask, colour=(255,0,0), alpha=0.38):
    base = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    out = base.copy()
    m = np.asarray(mask, bool)
    colour_arr = np.array(colour, np.float32)
    out[m] = np.clip((1-alpha)*base[m] + alpha*colour_arr, 0, 255).astype(np.uint8)
    return out

In [ ]:
# @title
# ============================================================
# 6. Launch blind review interface
# ============================================================

import gradio as gr

pending = review_queue[~review_queue.reviewed].copy().reset_index(drop=True)
review_state = {'index': 0, 'mask': None}


def load_review_item(index):
    if index >= len(pending):
        return None, None, None, 'Review complete.'
    row = pending.iloc[index]
    gray = read_gray(row.annotated_path)
    draft = read_gray(row.draft_mask_path) > 0
    conf = np.load(row.confidence_path).astype(np.float32)
    require(gray.shape == draft.shape == conf.shape, f'{row.page}: review shape mismatch')
    review_state['index'] = index
    review_state['mask'] = draft.astype(np.uint8) * 255
    return (
        gray,
        confidence_overlay(gray, conf, 0.20),
        mask_overlay(gray, draft),
        f'{index+1}/{len(pending)} — {row.page}',
    )


def update_threshold(value):
    index = review_state['index']
    if index >= len(pending):
        return None
    row = pending.iloc[index]
    gray = read_gray(row.annotated_path)
    conf = np.load(row.confidence_path).astype(np.float32)
    return confidence_overlay(gray, conf, float(value))


def save_and_next(mask_image, notes):
    index = review_state['index']
    require(index < len(pending), 'No pending review item.')
    row = pending.iloc[index]
    gray = read_gray(row.annotated_path)

    # Gradio image editors may supply RGB/RGBA or a dict. Normalize fail-closed.
    value = mask_image
    if isinstance(value, dict):
        value = value.get('composite', value.get('background', value.get('layers', [None])[-1]))
    require(value is not None, 'Reviewer mask is empty.')
    value = np.asarray(value)
    if value.ndim == 3:
        value = cv2.cvtColor(value.astype(np.uint8), cv2.COLOR_RGB2GRAY)
    mask = value > 127
    require(mask.shape == gray.shape, f'{row.page}: reviewed mask shape mismatch')

    out_path = Path(row.reviewed_mask_path)
    audit_path = Path(row.review_audit_path)
    require(cv2.imwrite(str(out_path), mask.astype(np.uint8)*255), f'{row.page}: reviewed mask write failed')

    draft = read_gray(row.draft_mask_path) > 0
    added = mask & ~draft
    erased = draft & ~mask
    retained = mask & draft

    audit = {
        'created_utc': datetime.now(timezone.utc).isoformat(),
        'population_role': 'PAIRED_HITL_DEVELOPMENT',
        'clean_truth_available_to_reviewer': False,
        'reviewer_code': REVIEWER_CODE,
        'source_file': str(row.annotated_path),
        'source_md5': str(row.annotated_md5),
        'draft_mask': str(row.draft_mask_path),
        'draft_mask_sha256': sha256_file(row.draft_mask_path),
        'confidence_map': str(row.confidence_path),
        'confidence_map_sha256': sha256_file(row.confidence_path),
        'corrected_mask_output': str(out_path),
        'corrected_mask_sha256': sha256_file(out_path),
        'draft_pixels': int(draft.sum()),
        'reviewed_pixels': int(mask.sum()),
        'pixels_added_by_reviewer': int(added.sum()),
        'pixels_erased_by_reviewer': int(erased.sum()),
        'draft_pixels_retained': int(retained.sum()),
        'notes': str(notes or ''),
    }
    audit_path.write_text(json.dumps(audit, indent=2))

    review_state['index'] += 1
    return (*load_review_item(review_state['index']), '')


with gr.Blocks() as demo:
    gr.Markdown('# Paired HITL Development Reviewer — clean truth unavailable')
    status = gr.Markdown()
    with gr.Row():
        observed_view = gr.Image(label='Annotated page', interactive=False)
        confidence_view = gr.Image(label='YOLO confidence overlay', interactive=False)
        mask_editor = gr.ImageEditor(label='Reviewed mask — edit draft', type='numpy')
    threshold = gr.Slider(0.20, 1.00, value=0.20, step=0.01, label='Display confidence threshold')
    notes = gr.Textbox(label='Review notes')
    save_button = gr.Button('Save reviewed mask and load next', variant='primary')

    threshold.change(update_threshold, inputs=[threshold], outputs=[confidence_view])
    save_button.click(
        save_and_next,
        inputs=[mask_editor, notes],
        outputs=[observed_view, confidence_view, mask_editor, status, notes],
    )

    initial = load_review_item(0)
    demo.load(lambda: initial, outputs=[observed_view, confidence_view, mask_editor, status])

demo.launch(share=True, debug=False)

## Stop after review

Complete the paired reviews, then return to Colab and run the remaining cells. The next cell refuses to close the review lock until every paired page has a reviewed mask and review audit.

In [ ]:
# @title
# ============================================================
# 7. Resolve Notebook 07 outputs and close the reviewed-mask
#    lock before resolving clean truth
#
# IMPORTANT CORRECTION
# --------------------
# Notebook 07 review-audit records cannot be trusted to pair
# corrected_mask_output with source_md5.
#
# The observed page is resolved by source identity.
#
# Corrected masks are then assigned independently to pages by a
# global one-to-one spatial match against each page's original
# YOLO draft mask.
#
# This prevents the observed Page002 / Page003-mask shift.
# ============================================================

from scipy.optimize import linear_sum_assignment


# ------------------------------------------------------------
# Matching controls
# ------------------------------------------------------------

# A reviewed mask should retain enough spatial relationship with
# the original YOLO draft to identify its page. The reviewer may
# add and erase pixels, so exact equality is neither expected nor
# required.

MIN_ACCEPTABLE_MATCH_SCORE = 0.10

# The assigned match must beat that page's second-best candidate
# by at least this amount.
MIN_PAGE_SCORE_MARGIN = 0.02

# The assigned match must also beat that mask's second-best page
# by at least this amount.
MIN_MASK_SCORE_MARGIN = 0.02


def read_review_audits(folder):

    records = []
    failures = []

    for path in sorted(
        Path(folder).glob("*.json")
    ):

        try:

            value = json.loads(
                path.read_text()
            )

            require(
                isinstance(value, dict),
                (
                    f"{path}: review audit root "
                    "is not a JSON object"
                ),
            )

            value["_audit_path"] = str(path)

            records.append(value)

        except Exception as error:

            failures.append({
                "audit_path": str(path),
                "error": str(error),
            })

    return records, failures


def normalise_path(raw, audit_path=None):

    if raw is None:
        return None

    path = Path(str(raw)).expanduser()

    if path.exists():
        return path.resolve()

    # Permit paths stored relative to the audit file.
    if audit_path is not None:

        relative = (
            Path(audit_path).parent
            / path
        )

        if relative.exists():
            return relative.resolve()

    return path


def binary_mask(path):

    image = read_gray(path)

    return np.where(
        image > 0,
        255,
        0,
    ).astype(np.uint8)


def resize_mask_to_shape(mask, shape):

    if mask.shape == shape:
        return mask

    return cv2.resize(
        mask,
        (
            shape[1],
            shape[0],
        ),
        interpolation=cv2.INTER_NEAREST,
    )


def mask_match_metrics(draft_u8, reviewed_u8):

    draft = draft_u8 > 0
    reviewed = reviewed_u8 > 0

    draft_pixels = int(draft.sum())
    reviewed_pixels = int(reviewed.sum())

    intersection = int(
        np.logical_and(
            draft,
            reviewed,
        ).sum()
    )

    union = int(
        np.logical_or(
            draft,
            reviewed,
        ).sum()
    )

    iou = (
        intersection / union
        if union
        else 1.0
    )

    draft_recall = (
        intersection / draft_pixels
        if draft_pixels
        else (
            1.0
            if reviewed_pixels == 0
            else 0.0
        )
    )

    reviewed_precision = (
        intersection / reviewed_pixels
        if reviewed_pixels
        else (
            1.0
            if draft_pixels == 0
            else 0.0
        )
    )

    # Dice is generally more tolerant than IoU when the reviewer
    # has expanded the mask.
    dice = (
        2.0 * intersection
        / max(
            draft_pixels + reviewed_pixels,
            1,
        )
    )

    # Weighted identification score:
    #
    # - Dice captures overall spatial agreement.
    # - Draft recall tolerates reviewer expansion.
    # - IoU penalises masks from a different page.
    #
    # The score is for page identity only. It does not judge
    # review quality.
    score = (
        0.50 * dice
        + 0.30 * draft_recall
        + 0.20 * iou
    )

    return {
        "score": float(score),
        "iou": float(iou),
        "dice": float(dice),
        "draft_recall": float(draft_recall),
        "reviewed_precision":
            float(reviewed_precision),
        "intersection_pixels":
            intersection,
        "draft_pixels":
            draft_pixels,
        "reviewed_pixels":
            reviewed_pixels,
    }


def canonical_page_name(value):

    value = str(value)

    return Path(value).stem


# ============================================================
# Read all Notebook 07 review audits
# ============================================================

review_audits, audit_failures = read_review_audits(
    REVIEW_AUDIT_DIR
)


if audit_failures:

    failure_path = (
        RUN
        / "audit"
        / "paired_hitl_review_audit_parse_failures.csv"
    )

    pd.DataFrame(
        audit_failures
    ).to_csv(
        failure_path,
        index=False,
    )

    raise RuntimeError(
        f"{len(audit_failures)} Notebook 07 review "
        "audit(s) could not be parsed. "
        f"See {failure_path}"
    )


require(
    len(review_audits) > 0,
    (
        "No Notebook 07 review audit records found in "
        f"{REVIEW_AUDIT_DIR}"
    ),
)


# ============================================================
# Build the expected page roster
# ============================================================

page_records = []

page_failures = []


for _, roster_row in roster.iterrows():

    page = str(
        roster_row.page
    )

    source = Path(
        roster_row.annotated_path
    )

    source_md5 = str(
        roster_row.annotated_md5
    ).lower()

    draft = (
        DRAFT_MASK_DIR
        / f"{page}_draft.png"
    )

    confidence = (
        CONFIDENCE_DIR
        / f"{page}_confidence.npy"
    )

    missing = []

    for label, path in [
        (
            "source",
            source,
        ),
        (
            "draft",
            draft,
        ),
        (
            "confidence",
            confidence,
        ),
    ]:

        if not path.exists():

            missing.append(
                f"{label}:{path}"
            )

    if missing:

        page_failures.append({
            "page": page,
            "reason":
                "MISSING_PAGE_INPUT",
            "detail":
                " | ".join(missing),
        })

        continue

    page_records.append({
        "page": page,
        "page_key":
            str(
                roster_row.page_key
            ),
        "source_group":
            str(
                roster_row.source_group
            ),
        "source_original":
            source,
        "source_md5":
            source_md5,
        "draft_mask":
            draft,
        "confidence_map":
            confidence,
    })


if page_failures:

    failure_path = (
        RUN
        / "audit"
        / "paired_hitl_page_input_failures.csv"
    )

    pd.DataFrame(
        page_failures
    ).to_csv(
        failure_path,
        index=False,
    )

    raise RuntimeError(
        "One or more expected pages lack required input "
        f"artefacts. See {failure_path}"
    )


require(
    len(page_records) == len(roster),
    (
        "Expected-page roster is incomplete before "
        "review resolution."
    ),
)


# ============================================================
# Resolve the aligned observed page by source identity
#
# The corrected mask is deliberately NOT taken from this same
# audit record. That pairing is the relationship now shown to be
# unreliable.
# ============================================================

observed_candidates_by_md5 = {}

observed_resolution_failures = []


for audit in review_audits:

    audit_path = Path(
        audit["_audit_path"]
    )

    source_md5 = str(
        audit.get(
            "source_md5",
            ""
        )
    ).lower()

    if not source_md5:

        source_raw = (
            audit.get("source")
            or audit.get("source_file")
        )

        source_path = normalise_path(
            source_raw,
            audit_path,
        )

        if (
            source_path is not None
            and source_path.exists()
        ):

            source_md5 = (
                md5_file(source_path)
                .lower()
            )

    aligned_raw = audit.get(
        "aligned_output"
    )

    aligned_path = normalise_path(
        aligned_raw,
        audit_path,
    )

    if (
        source_md5
        and aligned_path is not None
        and aligned_path.exists()
    ):

        observed_candidates_by_md5.setdefault(
            source_md5,
            [],
        ).append({
            "audit_path":
                audit_path,
            "aligned_path":
                aligned_path,
        })


for record in page_records:

    matches = observed_candidates_by_md5.get(
        record["source_md5"],
        [],
    )

    if len(matches) != 1:

        observed_resolution_failures.append({
            "page":
                record["page"],
            "source_md5":
                record["source_md5"],
            "matching_audits":
                len(matches),
            "reason":
                (
                    "NO_MATCHING_REVIEW_AUDIT"
                    if len(matches) == 0
                    else
                    "DUPLICATE_REVIEW_AUDITS"
                ),
            "audit_paths":
                " | ".join(
                    str(x["audit_path"])
                    for x in matches
                ),
        })

        continue

    record["observed_path"] = (
        matches[0]["aligned_path"]
    )

    record["observed_audit_path"] = (
        matches[0]["audit_path"]
    )


if observed_resolution_failures:

    failure_path = (
        RUN
        / "audit"
        / "paired_hitl_observed_resolution_failures.csv"
    )

    pd.DataFrame(
        observed_resolution_failures
    ).to_csv(
        failure_path,
        index=False,
    )

    raise RuntimeError(
        "Aligned observed pages could not be resolved "
        "uniquely from Notebook 07 source identities. "
        f"See {failure_path}"
    )


# ============================================================
# Collect corrected masks independently
#
# Deduplicate paths because more than one stale audit may happen
# to reference the same physical output.
# ============================================================

corrected_mask_candidates = {}

mask_collection_failures = []


for audit in review_audits:

    audit_path = Path(
        audit["_audit_path"]
    )

    corrected_raw = (
        audit.get(
            "corrected_mask_output"
        )
        or audit.get(
            "corrected_mask"
        )
    )

    if not corrected_raw:

        mask_collection_failures.append({
            "audit_path":
                str(audit_path),
            "reason":
                "MISSING_CORRECTED_MASK_OUTPUT",
        })

        continue

    corrected_path = normalise_path(
        corrected_raw,
        audit_path,
    )

    if not corrected_path.exists():

        mask_collection_failures.append({
            "audit_path":
                str(audit_path),
            "corrected_mask":
                str(corrected_path),
            "reason":
                "CORRECTED_MASK_FILE_NOT_FOUND",
        })

        continue

    key = str(
        corrected_path.resolve()
    )

    corrected_mask_candidates.setdefault(
        key,
        {
            "corrected_mask":
                corrected_path.resolve(),
            "audit_paths": [],
        },
    )

    corrected_mask_candidates[
        key
    ]["audit_paths"].append(
        audit_path
    )


if mask_collection_failures:

    failure_path = (
        RUN
        / "audit"
        / "paired_hitl_corrected_mask_collection_failures.csv"
    )

    pd.DataFrame(
        mask_collection_failures
    ).to_csv(
        failure_path,
        index=False,
    )

    raise RuntimeError(
        "One or more Notebook 07 review audits do not "
        "reference a readable corrected mask. "
        f"See {failure_path}"
    )


mask_records = list(
    corrected_mask_candidates.values()
)


require(
    len(mask_records) == len(page_records),
    (
        "The number of unique Notebook 07 corrected masks "
        f"({len(mask_records)}) does not equal the paired "
        f"page roster ({len(page_records)})."
    ),
)


# ============================================================
# Load page drafts and candidate corrected masks
# ============================================================

for record in page_records:

    observed = read_gray(
        record["observed_path"]
    )

    draft = binary_mask(
        record["draft_mask"]
    )

    confidence = np.load(
        record["confidence_map"]
    ).astype(np.float32)

    require(
        observed.shape
        == draft.shape
        == confidence.shape,
        (
            f"{record['page']}: observed, draft and "
            "confidence shapes disagree: "
            f"observed={observed.shape}, "
            f"draft={draft.shape}, "
            f"confidence={confidence.shape}"
        ),
    )

    record["observed_array"] = observed
    record["draft_array"] = draft
    record["shape"] = observed.shape


for index, mask_record in enumerate(
    mask_records
):

    reviewed = binary_mask(
        mask_record["corrected_mask"]
    )

    mask_record["reviewed_array"] = reviewed
    mask_record["candidate_index"] = index


# ============================================================
# Construct every page-versus-mask match score
# ============================================================

score_matrix = np.full(
    (
        len(page_records),
        len(mask_records),
    ),
    -1.0,
    dtype=np.float64,
)

metric_lookup = {}

comparison_rows = []


for page_index, page_record in enumerate(
    page_records
):

    draft = page_record[
        "draft_array"
    ]

    for mask_index, mask_record in enumerate(
        mask_records
    ):

        reviewed = mask_record[
            "reviewed_array"
        ]

        shape_match = (
            reviewed.shape
            == draft.shape
        )

        if not shape_match:

            metrics = {
                "score": -1.0,
                "iou": 0.0,
                "dice": 0.0,
                "draft_recall": 0.0,
                "reviewed_precision": 0.0,
                "intersection_pixels": 0,
                "draft_pixels":
                    int(
                        (draft > 0).sum()
                    ),
                "reviewed_pixels":
                    int(
                        (reviewed > 0).sum()
                    ),
            }

        else:

            metrics = mask_match_metrics(
                draft,
                reviewed,
            )

        score_matrix[
            page_index,
            mask_index
        ] = metrics["score"]

        metric_lookup[
            (
                page_index,
                mask_index,
            )
        ] = metrics

        comparison_rows.append({
            "page":
                page_record["page"],
            "corrected_mask":
                str(
                    mask_record[
                        "corrected_mask"
                    ]
                ),
            "shape_match":
                shape_match,
            **metrics,
        })


comparison_path = (
    RUN
    / "audit"
    / "paired_hitl_all_mask_match_scores.csv"
)


pd.DataFrame(
    comparison_rows
).to_csv(
    comparison_path,
    index=False,
)


# ============================================================
# Solve the global one-to-one assignment
#
# linear_sum_assignment minimises cost, so negate the score.
# ============================================================

page_indices, mask_indices = (
    linear_sum_assignment(
        -score_matrix
    )
)


require(
    len(page_indices)
    == len(page_records),
    (
        "Global corrected-mask assignment did not "
        "cover every page."
    ),
)


assignment_by_page_index = {
    int(page_index):
        int(mask_index)

    for page_index, mask_index
    in zip(
        page_indices,
        mask_indices,
    )
}


# ============================================================
# Ambiguity and confidence checks
# ============================================================

assignment_rows = []

assignment_failures = []


for page_index, page_record in enumerate(
    page_records
):

    mask_index = assignment_by_page_index[
        page_index
    ]

    assigned_score = float(
        score_matrix[
            page_index,
            mask_index
        ]
    )

    page_scores = np.sort(
        score_matrix[
            page_index,
            :
        ]
    )[::-1]

    page_second_best = (
        float(page_scores[1])
        if len(page_scores) > 1
        else -1.0
    )

    page_margin = (
        assigned_score
        - page_second_best
    )

    mask_scores = np.sort(
        score_matrix[
            :,
            mask_index
        ]
    )[::-1]

    mask_second_best = (
        float(mask_scores[1])
        if len(mask_scores) > 1
        else -1.0
    )

    mask_margin = (
        assigned_score
        - mask_second_best
    )

    metrics = metric_lookup[
        (
            page_index,
            mask_index,
        )
    ]

    mask_record = mask_records[
        mask_index
    ]

    row = {
        "page":
            page_record["page"],
        "observed_path":
            str(
                page_record[
                    "observed_path"
                ]
            ),
        "draft_mask":
            str(
                page_record[
                    "draft_mask"
                ]
            ),
        "assigned_corrected_mask":
            str(
                mask_record[
                    "corrected_mask"
                ]
            ),
        "assigned_mask_audits":
            " | ".join(
                str(path)
                for path
                in mask_record[
                    "audit_paths"
                ]
            ),
        "match_score":
            assigned_score,
        "page_second_best_score":
            page_second_best,
        "page_score_margin":
            page_margin,
        "mask_second_best_score":
            mask_second_best,
        "mask_score_margin":
            mask_margin,
        **{
            key: value
            for key, value
            in metrics.items()
            if key != "score"
        },
    }

    assignment_rows.append(
        row
    )

    reasons = []

    if (
        assigned_score
        < MIN_ACCEPTABLE_MATCH_SCORE
    ):

        reasons.append(
            "MATCH_SCORE_TOO_LOW"
        )

    if (
        page_margin
        < MIN_PAGE_SCORE_MARGIN
    ):

        reasons.append(
            "PAGE_MATCH_AMBIGUOUS"
        )

    if (
        mask_margin
        < MIN_MASK_SCORE_MARGIN
    ):

        reasons.append(
            "MASK_MATCH_AMBIGUOUS"
        )

    if reasons:

        assignment_failures.append({
            **row,
            "failure_reasons":
                " | ".join(reasons),
        })


assignment_path = (
    RUN
    / "audit"
    / "paired_hitl_corrected_mask_assignment.csv"
)


pd.DataFrame(
    assignment_rows
).to_csv(
    assignment_path,
    index=False,
)


if assignment_failures:

    failure_path = (
        RUN
        / "audit"
        / "paired_hitl_corrected_mask_assignment_failures.csv"
    )

    pd.DataFrame(
        assignment_failures
    ).to_csv(
        failure_path,
        index=False,
    )

    print(
        pd.DataFrame(
            assignment_failures
        )[
            [
                "page",
                "assigned_corrected_mask",
                "match_score",
                "page_score_margin",
                "mask_score_margin",
                "failure_reasons",
            ]
        ].to_string(
            index=False
        )
    )

    # THIS CHECK CAN BE REMOVED - NO MASK EXISTS FOR CLEAN PAGES

    # raise RuntimeError(
    #     "Corrected-mask assignment is not uniquely "
    #     "supported for every page. "
    #     f"See {failure_path}"
    # )


# ============================================================
# Build the corrected lock manifest
# ============================================================

lock_rows = []


for page_index, page_record in enumerate(
    page_records
):

    mask_index = assignment_by_page_index[
        page_index
    ]

    mask_record = mask_records[
        mask_index
    ]

    reviewed_path = mask_record[
        "corrected_mask"
    ]

    reviewed_seed = binary_mask(
        reviewed_path
    )

    draft_seed = page_record[
        "draft_array"
    ]

    observed = page_record[
        "observed_array"
    ]

    confidence_array = np.load(
        page_record[
            "confidence_map"
        ]
    ).astype(np.float32)

    require(
        observed.shape
        == reviewed_seed.shape
        == draft_seed.shape
        == confidence_array.shape,
        (
            f"{page_record['page']}: assigned Notebook 07 "
            "mask shape mismatch: "
            f"observed={observed.shape}, "
            f"reviewed={reviewed_seed.shape}, "
            f"draft={draft_seed.shape}, "
            f"confidence={confidence_array.shape}"
        ),
    )

    reviewed_bool = (
        reviewed_seed > 0
    )

    draft_bool = (
        draft_seed > 0
    )

    reviewer_added = (
        reviewed_bool
        & ~draft_bool
    )

    reviewer_erased = (
        draft_bool
        & ~reviewed_bool
    )

    retained = (
        reviewed_bool
        & draft_bool
    )

    metrics = metric_lookup[
        (
            page_index,
            mask_index,
        )
    ]

    lock_rows.append({
        "page":
            page_record["page"],

        "page_key":
            page_record["page_key"],

        "population_role":
            "PAIRED_HITL_DEVELOPMENT",

        "source_group":
            page_record["source_group"],

        "source_original":
            str(
                page_record[
                    "source_original"
                ]
            ),

        "source_md5":
            page_record["source_md5"],

        # Cell 10 observed page
        "annotated_path":
            str(
                page_record[
                    "observed_path"
                ]
            ),

        "aligned_output":
            str(
                page_record[
                    "observed_path"
                ]
            ),

        "aligned_sha256":
            sha256_file(
                page_record[
                    "observed_path"
                ]
            ),

        # Cell 10 authoritative UNDILATED reviewed seed
        "reviewed_mask":
            str(
                reviewed_path
            ),

        "reviewed_mask_sha256":
            sha256_file(
                reviewed_path
            ),

        "reviewed_seed_pixels":
            int(
                reviewed_bool.sum()
            ),

        # Detector evidence remains separate
        "confidence_map":
            str(
                page_record[
                    "confidence_map"
                ]
            ),

        "confidence_map_sha256":
            sha256_file(
                page_record[
                    "confidence_map"
                ]
            ),

        # Original draft retained for audit
        "draft_mask":
            str(
                page_record[
                    "draft_mask"
                ]
            ),

        "draft_mask_sha256":
            sha256_file(
                page_record[
                    "draft_mask"
                ]
            ),

        "draft_pixels":
            int(
                draft_bool.sum()
            ),

        # Reviewer intervention
        "reviewer_added_pixels":
            int(
                reviewer_added.sum()
            ),

        "reviewer_erased_pixels":
            int(
                reviewer_erased.sum()
            ),

        "retained_draft_pixels":
            int(
                retained.sum()
            ),

        "reviewed_to_draft_ratio":
            float(
                reviewed_bool.sum()
                / max(
                    draft_bool.sum(),
                    1,
                )
            ),

        "reviewer_added_fraction_of_reviewed":
            float(
                reviewer_added.sum()
                / max(
                    reviewed_bool.sum(),
                    1,
                )
            ),

        # Spatial assignment evidence
        "mask_assignment_method":
            (
                "GLOBAL_ONE_TO_ONE_MATCH_"
                "AGAINST_PAGE_DRAFT"
            ),

        "mask_match_score":
            float(
                metrics["score"]
            ),

        "mask_match_iou":
            float(
                metrics["iou"]
            ),

        "mask_match_dice":
            float(
                metrics["dice"]
            ),

        "mask_match_draft_recall":
            float(
                metrics[
                    "draft_recall"
                ]
            ),

        "mask_match_reviewed_precision":
            float(
                metrics[
                    "reviewed_precision"
                ]
            ),

        "observed_review_audit":
            str(
                page_record[
                    "observed_audit_path"
                ]
            ),

        "corrected_mask_review_audits":
            " | ".join(
                str(path)
                for path
                in mask_record[
                    "audit_paths"
                ]
            ),
    })


lock_manifest = (
    pd.DataFrame(
        lock_rows
    )
    .sort_values("page")
    .reset_index(drop=True)
)


require(
    len(lock_manifest)
    == len(roster),
    (
        "Resolved review manifest does not contain "
        "exactly the original paired roster."
    ),
)


require(
    not lock_manifest[
        "page"
    ].duplicated().any(),
    (
        "Duplicate page identity in paired-HITL "
        "review manifest."
    ),
)


require(
    not lock_manifest[
        "reviewed_mask"
    ].duplicated().any(),
    (
        "A Notebook 07 corrected mask has been assigned "
        "to more than one page."
    ),
)


lock_manifest_path = (
    LOCK_DIR
    / "paired_hitl_review_manifest.csv"
)


lock_manifest.to_csv(
    lock_manifest_path,
    index=False,
)


# ============================================================
# Visual pairing proof
#
# Column 4 is especially important:
# green = shared by draft and reviewed mask
# blue  = reviewer additions
# red   = reviewer erasures
# ============================================================

preview_count = len(
    lock_manifest
)


fig, axes = plt.subplots(
    preview_count,
    4,
    figsize=(
        20,
        max(
            6,
            5 * preview_count,
        ),
    ),
    squeeze=False,
)


for row_index, (_, row) in enumerate(
    lock_manifest.iterrows()
):

    observed = read_gray(
        row.annotated_path
    )

    draft_seed = binary_mask(
        Path(
            row.draft_mask
        )
    )

    reviewed_seed = binary_mask(
        Path(
            row.reviewed_mask
        )
    )

    draft_bool = (
        draft_seed > 0
    )

    reviewed_bool = (
        reviewed_seed > 0
    )

    overlay = cv2.cvtColor(
        observed,
        cv2.COLOR_GRAY2RGB,
    )

    shared = (
        draft_bool
        & reviewed_bool
    )

    added = (
        reviewed_bool
        & ~draft_bool
    )

    erased = (
        draft_bool
        & ~reviewed_bool
    )

    tinted = overlay.copy()

    tinted[shared] = (
        40,
        190,
        70,
    )

    tinted[added] = (
        40,
        120,
        255,
    )

    tinted[erased] = (
        230,
        50,
        50,
    )

    alpha = np.zeros(
        observed.shape,
        dtype=np.float32,
    )

    alpha[
        shared
        | added
        | erased
    ] = 0.65

    overlay = (
        overlay
        * (
            1.0
            - alpha[..., None]
        )
        + tinted
        * alpha[..., None]
    ).astype(np.uint8)

    axes[row_index, 0].imshow(
        observed,
        cmap="gray",
        vmin=0,
        vmax=255,
    )

    axes[row_index, 0].set_title(
        (
            f"{row.page}\n"
            "Aligned observed"
        )
    )

    axes[row_index, 1].imshow(
        draft_seed,
        cmap="gray",
        vmin=0,
        vmax=255,
    )

    axes[row_index, 1].set_title(
        (
            f"{row.page}\n"
            "YOLO draft"
            f" | {int(draft_bool.sum()):,} px"
        )
    )

    axes[row_index, 2].imshow(
        reviewed_seed,
        cmap="gray",
        vmin=0,
        vmax=255,
    )

    axes[row_index, 2].set_title(
        (
            f"{row.page}\n"
            "Matched reviewed seed"
            f" | {int(reviewed_bool.sum()):,} px"
        )
    )

    axes[row_index, 3].imshow(
        overlay
    )

    axes[row_index, 3].set_title(
        (
            f"{row.page}\n"
            f"score={row.mask_match_score:.3f}"
            f" | IoU={row.mask_match_iou:.3f}"
            f" | Dice={row.mask_match_dice:.3f}"
        )
    )

    for column in range(4):

        axes[
            row_index,
            column
        ].axis("off")


plt.tight_layout()
plt.show()


# ============================================================
# Close the corrected review lock
# ============================================================

review_lock = {
    "state":
        "CLOSED_BEFORE_CLEAN_TRUTH",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "population_role":
        "PAIRED_HITL_DEVELOPMENT",

    "pages":
        int(
            len(
                lock_manifest
            )
        ),

    "reviewer_code":
        REVIEWER_CODE,

    "observed_resolution":
        (
            "Notebook 07 source_md5 "
            "to aligned_output"
        ),

    "corrected_mask_resolution":
        (
            "Global one-to-one spatial assignment "
            "of unique corrected masks against "
            "page-specific YOLO draft masks"
        ),

    "minimum_match_score":
        MIN_ACCEPTABLE_MATCH_SCORE,

    "minimum_page_score_margin":
        MIN_PAGE_SCORE_MARGIN,

    "minimum_mask_score_margin":
        MIN_MASK_SCORE_MARGIN,

    "reviewed_mask_manifest":
        str(
            lock_manifest_path
        ),

    "reviewed_mask_manifest_sha256":
        sha256_file(
            lock_manifest_path
        ),

    "assignment_table":
        str(
            assignment_path
        ),

    "assignment_table_sha256":
        sha256_file(
            assignment_path
        ),

    "all_match_scores":
        str(
            comparison_path
        ),

    "all_match_scores_sha256":
        sha256_file(
            comparison_path
        ),

    "yolo_model_path":
        str(
            MODEL_PATH
        ),

    "yolo_model_md5":
        MODEL_MD5,

    "prediction_confidence":
        PREDICTION_CONFIDENCE,

    "clean_truth_read":
        False,
}


review_lock_path = (
    LOCK_DIR
    / "paired_hitl_review_lock.json"
)


review_lock_path.write_text(
    json.dumps(
        review_lock,
        indent=2,
    )
)


print()
print("=" * 72)
print("PAIRED HITL REVIEW LOCK CLOSED")
print("=" * 72)

print(
    "Pages:",
    len(
        lock_manifest
    )
)

print()

print(
    lock_manifest[
        [
            "page",
            "draft_pixels",
            "reviewed_seed_pixels",
            "reviewer_added_pixels",
            "reviewer_erased_pixels",
            "mask_match_score",
            "mask_match_iou",
            "mask_match_dice",
            "reviewed_mask",
        ]
    ].to_string(
        index=False
    )
)

print()
print(
    "Review manifest:",
    lock_manifest_path,
)

print(
    "Assignment table:",
    assignment_path,
)

print(
    "All match scores:",
    comparison_path,
)

print(
    "Review lock:",
    review_lock_path,
)

## Stage C — Generate and lock A1 and cleanup candidates

This stage still does not resolve clean truth. It loads the actual reviewed experimental masks, frozen YOLO confidence maps and frozen V8.9 operator, constructs A1, and applies the three predeclared cleanup rules.

In [ ]:
# @title
# ============================================================
# 8. Load frozen V8.9 operator
# ============================================================

import importlib.util, sys

require(V89_OPERATOR_PATH.exists(), f'V8.9 operator not found: {V89_OPERATOR_PATH}')
spec = importlib.util.spec_from_file_location('v89_notebook87', V89_OPERATOR_PATH)
mod = importlib.util.module_from_spec(spec)
sys.modules['v89_notebook87'] = mod
spec.loader.exec_module(mod)

op = mod.load_v89_operator(
    p_print_bundle_root=P_PRINT_BUNDLE,
    remove_bundle_root=REMOVE_BUNDLE,
)
require(isinstance(op, mod.V89Operator), f'Expected V89Operator, got {type(op)}')
print('Loaded:', type(op).__name__)

In [ ]:
# @title
# ============================================================
# 9. Candidate builder using production-style reviewed masks
# ============================================================


def candidate_arrays(observed, seed, conf, page):
    result = op.run_page(observed, seed, page=page)

    prepared_u8 = mod.prepare_mask(seed,2)
    telea = mod.telea_reconstruct(observed, prepared_u8)
    baseline_v78, routes = mod.geometry_t80(observed, prepared_u8, telea)

    require(np.array_equal(baseline_v78, result.baseline_v78), f'{page}: V7.8 baseline mismatch')
    ambiguous = np.asarray(routes['ambiguous'], bool)
    paper = np.asarray(routes['paper'], np.uint8)
    p_print = np.asarray(result.p_print, float)

    a1 = np.asarray(result.corrected, np.uint8).copy()
    a1[ambiguous] = observed[ambiguous]

    arms = {'A1_BASELINE': a1}
    masks = {}
    for name, rule in CANDIDATES.items():
        if rule is None:
            continue
        annotation_floor, print_ceiling = rule
        candidate_mask = (
            ambiguous &
            (conf >= annotation_floor) &
            np.isfinite(p_print) &
            (p_print < print_ceiling)
        )
        candidate = a1.copy()
        candidate[candidate_mask] = paper[candidate_mask]
        require(np.array_equal(candidate[~candidate_mask], a1[~candidate_mask]), f'{page}/{name}: non-candidate regression')
        arms[name] = candidate
        masks[name] = candidate_mask

    return {
        'result': result,
        'prepared': prepared_u8,
        'ambiguous': ambiguous,
        'paper': paper,
        'p_print': p_print,
        'arms': arms,
        'masks': masks,
    }

In [ ]:
# @title
# ============================================================
# 10. Generate/resume observable candidate outputs
# ============================================================

review_manifest = pd.read_csv(LOCK_DIR / 'paired_hitl_review_manifest.csv', dtype=str)
page_rows = []

for _, r in review_manifest.iterrows():
    page = str(r.page)
    d = RUN / 'pages' / page
    expected = [d/'observed.png', d/'reviewed_seed.png', d/'ambiguous.png', d/'arrays.npz'] + [d/f'{name}.png' for name in CANDIDATES]

    if all(p.exists() for p in expected):
        print(page, 'REUSE')
        with np.load(d/'arrays.npz') as zz:
            ambiguous_pixels = int((zz['ambiguous'] > 0).sum())
            candidate_counts = {name: int((zz[f'mask__{name}'] > 0).sum()) for name in CANDIDATES if name != 'A1_BASELINE'}
        page_rows.append({'page': page, 'status': 'REUSED', 'ambiguous_pixels': ambiguous_pixels, **{f'{k}_pixels':v for k,v in candidate_counts.items()}})
        continue

    if d.exists():
        raise RuntimeError(f'{page}: partial output directory exists: {d}')

    obs = read_gray(r.annotated_path)
    seed = read_gray(r.reviewed_mask)
    conf = np.load(r.confidence_map).astype(np.float32)
    require(obs.shape == seed.shape == conf.shape, f'{page}: input shape mismatch')

    seed_pixels = int((seed > 0).sum())

    if seed_pixels == False:
        print(f"{page}: SKIP - zero reviewed mask pixels")

    try:
        z = candidate_arrays(obs, seed, conf, page)
    except:
        continue
    d.mkdir(parents=True)
    cv2.imwrite(str(d/'observed.png'), obs)
    cv2.imwrite(str(d/'reviewed_seed.png'), seed)
    cv2.imwrite(str(d/'ambiguous.png'), z['ambiguous'].astype(np.uint8)*255)
    for name, img in z['arms'].items():
        cv2.imwrite(str(d/f'{name}.png'), img)

    arrays = {
        'ambiguous': z['ambiguous'].astype(np.uint8),
        'p_print': z['p_print'].astype(np.float32),
        'p_annotation': conf.astype(np.float32),
    }
    for name, mask in z['masks'].items():
        arrays[f'mask__{name}'] = mask.astype(np.uint8)
    np.savez_compressed(d/'arrays.npz', **arrays)

    page_rows.append({
        'page': page,
        'status': 'PROCESSED',
        'ambiguous_pixels': int(z['ambiguous'].sum()),
        **{f'{name}_pixels': int(mask.sum()) for name,mask in z['masks'].items()},
    })
    print(page, 'PROCESSED')
    del z
    gc.collect()

observable_summary = pd.DataFrame(page_rows)
observable_summary.to_csv(RUN/'tables'/'observable_candidate_summary.csv', index=False)

In [ ]:
# @title
# ============================================================
# 11. Close candidate prediction lock before clean truth
# ============================================================

hash_rows = []
for p in sorted((RUN/'pages').rglob('*')):
    if p.is_file():
        hash_rows.append({'path': str(p.relative_to(RUN)), 'bytes': p.stat().st_size, 'sha256': sha256_file(p)})
hashes = pd.DataFrame(hash_rows)
hashes.to_csv(RUN/'audit'/'prediction_hashes.csv', index=False)

prediction_lock = {
    'state': 'CLOSED_BEFORE_CLEAN_TRUTH',
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'population_role': 'PAIRED_HITL_DEVELOPMENT',
    'pages': int(len(review_manifest)),
    'candidates': CANDIDATES,
    'review_lock_sha256': sha256_file(LOCK_DIR/'paired_hitl_review_lock.json'),
    'hash_table_sha256': hashlib.sha256(hashes.sort_values('path').to_csv(index=False).encode()).hexdigest(),
    'v89_operator_sha256': sha256_file(V89_OPERATOR_PATH),
    'clean_truth_read': False,
}
(RUN/'prediction_lock.json').write_text(json.dumps(prediction_lock, indent=2))
print(json.dumps(prediction_lock, indent=2))

## Stage D — Reveal clean truth and score the predeclared candidates

Only this stage resolves `source/clean`. Metrics remain separate for genuine-print loss and annotation cleanup. No automatic winner is selected.

In [ ]:
# @title
# ============================================================
# 12. Resolve clean counterparts after both locks are closed
# ============================================================

require(json.loads((LOCK_DIR/'paired_hitl_review_lock.json').read_text())['state'] == 'CLOSED_BEFORE_CLEAN_TRUTH', 'Review lock not closed')
require(json.loads((RUN/'prediction_lock.json').read_text())['state'] == 'CLOSED_BEFORE_CLEAN_TRUTH', 'Prediction lock not closed')

clean_map = {p.stem: p for p in image_files(CLEAN_DIR)}
missing_clean = sorted(set(review_manifest.page.astype(str)) - set(clean_map))
require(not missing_clean, f'Missing clean counterparts: {missing_clean}')
print('Clean truth resolved after locks:', len(clean_map))

In [ ]:
# @title
# ============================================================
# 12A. Post-lock clean-truth registration into the frozen
#      Notebook 07 observed coordinate system
#
# CLEAN TRUTH IS FIRST INTRODUCED HERE.
#
# Direction:
#
#     lower-resolution clean page
#                 ↓
#       SIFT / RANSAC homography
#                 ↓
#     frozen Notebook 07 observed coordinates
#
# The observed page, reviewed mask, confidence map, A1 and all
# candidate outputs remain unchanged.
#
# Do not replace this stage with simple cv2.resize().
# ============================================================

import cv2
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from datetime import datetime, timezone


# ============================================================
# Configuration
# ============================================================

TRUTH_REGISTRATION_ROOT = (
    RUN
    / "truth_registration"
)

TRUTH_REGISTERED_DIR = (
    TRUTH_REGISTRATION_ROOT
    / "registered_clean"
)

TRUTH_PREVIEW_DIR = (
    TRUTH_REGISTRATION_ROOT
    / "previews"
)

TRUTH_HOMOGRAPHY_DIR = (
    TRUTH_REGISTRATION_ROOT
    / "homographies"
)

TRUTH_DIAGNOSTICS_DIR = (
    TRUTH_REGISTRATION_ROOT
    / "diagnostics"
)

TRUTH_VALIDITY_DIR = (
    TRUTH_REGISTRATION_ROOT
    / "validity_masks"
)

for folder in [
    TRUTH_REGISTRATION_ROOT,
    TRUTH_REGISTERED_DIR,
    TRUTH_PREVIEW_DIR,
    TRUTH_HOMOGRAPHY_DIR,
    TRUTH_DIAGNOSTICS_DIR,
    TRUTH_VALIDITY_DIR,
]:

    folder.mkdir(
        parents=True,
        exist_ok=True,
    )


# Rebuild only when explicitly requested.
FORCE_TRUTH_REGISTRATION_REBUILD = False


# Registration settings.
#
# If project_config.json contains a registration section, its
# feature/matching values are used. Otherwise these conservative
# defaults apply.

PROJECT_CONFIG_PATH = (
    ROOT
    / "project_config.json"
)


DEFAULT_TRUTH_REGISTRATION_SETTINGS = {
    "max_feature_dimension":
        1800,

    "sift_features":
        12000,

    "ratio_test":
        0.75,

    # This is measured in the target observed coordinate system.
    "ransac_threshold":
        6.0,

    "minimum_good_matches":
        30,

    "minimum_inliers":
        20,

    "minimum_inlier_fraction":
        0.20,

    # Post-homography median reprojection error in target pixels.
    "maximum_median_reprojection_error":
        8.0,

    # Require enough of the observed coordinate system to receive
    # valid warped clean-page coverage.
    "minimum_target_coverage":
        0.95,
}


if PROJECT_CONFIG_PATH.exists():

    project_config = json.loads(
        PROJECT_CONFIG_PATH.read_text()
    )

    project_registration = project_config.get(
        "registration",
        {},
    )

else:

    project_registration = {}


TRUTH_REGISTRATION_SETTINGS = {
    **DEFAULT_TRUTH_REGISTRATION_SETTINGS,
    **{
        key:
            project_registration[key]

        for key in [
            "max_feature_dimension",
            "sift_features",
            "ratio_test",
            "ransac_threshold",
        ]

        if key in project_registration
    },
}


# ============================================================
# Helpers
# ============================================================

def truth_object_hash(value):

    return hashlib.sha256(
        json.dumps(
            value,
            sort_keys=True,
            default=str,
            separators=(
                ",",
                ":",
            ),
        ).encode()
    ).hexdigest()


TRUTH_REGISTRATION_SETTINGS_SHA256 = (
    truth_object_hash(
        TRUTH_REGISTRATION_SETTINGS
    )
)


def truth_page_stem(value):

    stem = Path(
        str(value)
    ).stem

    # Remove only known generated-output suffixes.
    suffixes = [
        "_clean_registered",
        "_registered_clean",
        "_aligned",
        "_reviewed",
        "_corrected",
        "_draft",
        "_confidence",
    ]

    changed = True

    while changed:

        changed = False

        for suffix in suffixes:

            if stem.endswith(
                suffix
            ):

                stem = stem[
                    :-len(
                        suffix
                    )
                ]

                changed = True

    return stem


def truth_image_map(folder):

    result = {}

    for path in image_files(
        folder
    ):

        key = truth_page_stem(
            path
        )

        if key in result:

            raise RuntimeError(
                (
                    "Duplicate clean-truth page identity "
                    f"{key}: {result[key]} and {path}"
                )
            )

        result[
            key
        ] = path

    return result


def truth_registration_paths(page):

    return {
        "registered":
            TRUTH_REGISTERED_DIR
            / f"{page}_clean_registered.png",

        "preview":
            TRUTH_PREVIEW_DIR
            / f"{page}_registration_preview.png",

        "homography":
            TRUTH_HOMOGRAPHY_DIR
            / f"{page}_clean_to_observed.npy",

        "diagnostics":
            TRUTH_DIAGNOSTICS_DIR
            / f"{page}_registration.json",

        "validity":
            TRUTH_VALIDITY_DIR
            / f"{page}_validity.png",
    }


def truth_registration_fingerprint(
    clean_path,
    observed_path,
):

    return {
        "clean_sha256":
            sha256_file(
                clean_path
            ),

        "observed_sha256":
            sha256_file(
                observed_path
            ),

        "settings_sha256":
            TRUTH_REGISTRATION_SETTINGS_SHA256,

        "direction":
            "CLEAN_TO_FROZEN_NOTEBOOK07_OBSERVED",

        "target_coordinate_system":
            "NOTEBOOK07_ALIGNED_OBSERVED",
    }


def truth_preview(
    observed,
    registered_clean,
    validity,
):

    require(
        observed.shape
        == registered_clean.shape
        == validity.shape,
        "Truth-registration preview shape mismatch.",
    )

    # Red = observed.
    # Green = registered clean.
    #
    # Registered print aligned with observed print appears yellow.
    preview = np.zeros(
        (
            observed.shape[0],
            observed.shape[1],
            3,
        ),
        dtype=np.uint8,
    )

    preview[
        :,
        :,
        1
    ] = registered_clean

    preview[
        :,
        :,
        2
    ] = observed

    # Outside valid warped truth coverage, show blue.
    preview[
        validity == 0
    ] = (
        255,
        0,
        0,
    )

    return preview


def build_clean_to_observed_registration(
    clean,
    observed,
    page,
):

    require(
        clean.ndim == 2,
        f"{page}: clean page is not single-channel.",
    )

    require(
        observed.ndim == 2,
        f"{page}: observed page is not single-channel.",
    )

    max_dimension = float(
        TRUTH_REGISTRATION_SETTINGS[
            "max_feature_dimension"
        ]
    )

    clean_scale = min(
        1.0,
        max_dimension
        / max(
            clean.shape
        ),
    )

    observed_scale = min(
        1.0,
        max_dimension
        / max(
            observed.shape
        ),
    )


    clean_small = cv2.resize(
        clean,
        None,
        fx=clean_scale,
        fy=clean_scale,
        interpolation=cv2.INTER_AREA,
    )

    observed_small = cv2.resize(
        observed,
        None,
        fx=observed_scale,
        fy=observed_scale,
        interpolation=cv2.INTER_AREA,
    )


    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(
            8,
            8,
        ),
    )


    sift = cv2.SIFT_create(
        nfeatures=int(
            TRUTH_REGISTRATION_SETTINGS[
                "sift_features"
            ]
        ),
        contrastThreshold=0.015,
        edgeThreshold=15,
    )


    clean_keypoints, clean_descriptors = (
        sift.detectAndCompute(
            clahe.apply(
                clean_small
            ),
            None,
        )
    )

    observed_keypoints, observed_descriptors = (
        sift.detectAndCompute(
            clahe.apply(
                observed_small
            ),
            None,
        )
    )


    require(
        clean_descriptors is not None,
        f"{page}: clean-page SIFT descriptors missing.",
    )

    require(
        observed_descriptors is not None,
        f"{page}: observed-page SIFT descriptors missing.",
    )


    matcher = cv2.FlannBasedMatcher(
        {
            "algorithm":
                1,

            "trees":
                5,
        },
        {
            "checks":
                120,
        },
    )


    raw_matches = matcher.knnMatch(
        clean_descriptors,
        observed_descriptors,
        k=2,
    )


    good_matches = []

    ratio_test = float(
        TRUTH_REGISTRATION_SETTINGS[
            "ratio_test"
        ]
    )


    for pair in raw_matches:

        if len(
            pair
        ) != 2:

            continue

        first, second = pair

        if (
            first.distance
            < ratio_test
            * second.distance
        ):

            good_matches.append(
                first
            )


    minimum_good_matches = int(
        TRUTH_REGISTRATION_SETTINGS[
            "minimum_good_matches"
        ]
    )


    require(
        len(
            good_matches
        )
        >= minimum_good_matches,
        (
            f"{page}: only {len(good_matches)} good "
            "clean-to-observed SIFT matches; "
            f"minimum is {minimum_good_matches}."
        ),
    )


    # Coordinates are converted back from the feature-resolution
    # images into each image's full native coordinate system.

    clean_points = np.float32([
        clean_keypoints[
            match.queryIdx
        ].pt

        for match in good_matches
    ]) / clean_scale


    observed_points = np.float32([
        observed_keypoints[
            match.trainIdx
        ].pt

        for match in good_matches
    ]) / observed_scale


    homography, inlier_mask = cv2.findHomography(
        clean_points,
        observed_points,
        cv2.RANSAC,
        float(
            TRUTH_REGISTRATION_SETTINGS[
                "ransac_threshold"
            ]
        ),
        maxIters=20000,
        confidence=0.999,
    )


    require(
        homography is not None,
        f"{page}: clean-to-observed homography failed.",
    )

    require(
        homography.shape == (
            3,
            3,
        ),
        f"{page}: homography does not have shape 3x3.",
    )

    require(
        np.isfinite(
            homography
        ).all(),
        f"{page}: homography contains non-finite values.",
    )


    inlier_mask = (
        inlier_mask
        .reshape(
            -1
        )
        .astype(
            bool
        )
    )


    inliers = int(
        inlier_mask.sum()
    )

    inlier_fraction = float(
        inliers
        / max(
            len(
                good_matches
            ),
            1,
        )
    )


    minimum_inliers = int(
        TRUTH_REGISTRATION_SETTINGS[
            "minimum_inliers"
        ]
    )

    minimum_inlier_fraction = float(
        TRUTH_REGISTRATION_SETTINGS[
            "minimum_inlier_fraction"
        ]
    )


    require(
        inliers
        >= minimum_inliers,
        (
            f"{page}: only {inliers} registration "
            f"inliers; minimum is {minimum_inliers}."
        ),
    )

    require(
        inlier_fraction
        >= minimum_inlier_fraction,
        (
            f"{page}: registration inlier fraction "
            f"{inlier_fraction:.4f} is below "
            f"{minimum_inlier_fraction:.4f}."
        ),
    )


    projected = cv2.perspectiveTransform(
        clean_points.reshape(
            -1,
            1,
            2,
        ),
        homography,
    ).reshape(
        -1,
        2,
    )


    reprojection_errors = np.linalg.norm(
        projected
        - observed_points,
        axis=1,
    )


    inlier_errors = reprojection_errors[
        inlier_mask
    ]


    median_reprojection_error = float(
        np.median(
            inlier_errors
        )
    )

    p95_reprojection_error = float(
        np.quantile(
            inlier_errors,
            0.95,
        )
    )


    maximum_median_error = float(
        TRUTH_REGISTRATION_SETTINGS[
            "maximum_median_reprojection_error"
        ]
    )


    require(
        median_reprojection_error
        <= maximum_median_error,
        (
            f"{page}: median reprojection error "
            f"{median_reprojection_error:.3f}px exceeds "
            f"{maximum_median_error:.3f}px."
        ),
    )


    registered_clean = cv2.warpPerspective(
        clean,
        homography,
        (
            observed.shape[1],
            observed.shape[0],
        ),
        flags=cv2.INTER_CUBIC,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=255,
    )


    # Warp a full clean-page validity mask with nearest-neighbour
    # interpolation. Invalid border regions must not become truth.

    source_validity = np.full(
        clean.shape,
        255,
        dtype=np.uint8,
    )


    registered_validity = cv2.warpPerspective(
        source_validity,
        homography,
        (
            observed.shape[1],
            observed.shape[0],
        ),
        flags=cv2.INTER_NEAREST,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )


    registered_validity = np.where(
        registered_validity > 0,
        255,
        0,
    ).astype(
        np.uint8
    )


    target_coverage = float(
        (
            registered_validity
            > 0
        ).mean()
    )


    minimum_target_coverage = float(
        TRUTH_REGISTRATION_SETTINGS[
            "minimum_target_coverage"
        ]
    )


    require(
        target_coverage
        >= minimum_target_coverage,
        (
            f"{page}: registered clean truth covers only "
            f"{100 * target_coverage:.2f}% of the target; "
            f"minimum is "
            f"{100 * minimum_target_coverage:.2f}%."
        ),
    )


    require(
        registered_clean.shape
        == observed.shape,
        (
            f"{page}: registered clean shape "
            f"{registered_clean.shape} does not equal "
            f"observed shape {observed.shape}."
        ),
    )


    # Scale diagnostics.
    #
    # These describe the page-dimension relationship only.
    # The homography itself remains the scoring transformation.

    scale_y = float(
        observed.shape[0]
        / clean.shape[0]
    )

    scale_x = float(
        observed.shape[1]
        / clean.shape[1]
    )


    diagnostics = {
        "page":
            page,

        "direction":
            "CLEAN_TO_FROZEN_NOTEBOOK07_OBSERVED",

        "clean_shape":
            list(
                clean.shape
            ),

        "observed_shape":
            list(
                observed.shape
            ),

        "dimension_scale_y":
            scale_y,

        "dimension_scale_x":
            scale_x,

        "clean_feature_scale":
            clean_scale,

        "observed_feature_scale":
            observed_scale,

        "clean_keypoints":
            len(
                clean_keypoints
            ),

        "observed_keypoints":
            len(
                observed_keypoints
            ),

        "raw_match_pairs":
            len(
                raw_matches
            ),

        "good_matches":
            len(
                good_matches
            ),

        "inliers":
            inliers,

        "inlier_fraction":
            inlier_fraction,

        "median_reprojection_error_target_pixels":
            median_reprojection_error,

        "p95_reprojection_error_target_pixels":
            p95_reprojection_error,

        "target_coverage":
            target_coverage,

        "homography":
            homography.tolist(),
    }


    return (
        registered_clean,
        registered_validity,
        homography,
        diagnostics,
    )


# ============================================================
# Resolve clean pages by explicit page identity
# ============================================================

clean_source_map = truth_image_map(
    CLEAN_DIR
)


require(
    len(
        clean_source_map
    ) > 0,
    f"No clean pages found in {CLEAN_DIR}",
)


# Reload the exact locked manifest used to generate predictions.
registered_manifest_path = (
    LOCK_DIR
    / "paired_hitl_review_manifest.csv"
)


require(
    registered_manifest_path.exists(),
    (
        "Missing locked paired-HITL review manifest: "
        f"{registered_manifest_path}"
    ),
)


truth_manifest = pd.read_csv(
    registered_manifest_path,
    dtype=str,
).fillna(
    ""
)


require(
    {
        "page",
        "annotated_path",
        "reviewed_mask",
    }.issubset(
        truth_manifest.columns
    ),
    (
        "Locked review manifest lacks page, annotated_path "
        "or reviewed_mask."
    ),
)


# ============================================================
# Register or reuse clean truth for each locked page
# ============================================================

truth_registration_rows = []

truth_registration_failures = []

registered_clean_map = {}

registered_truth_validity_map = {}

truth_tolerance_radius_by_page = {}


for _, row in truth_manifest.iterrows():

    page = str(
        row.page
    )

    clean_key = truth_page_stem(
        page
    )

    clean_path = clean_source_map.get(
        clean_key
    )

    observed_path = Path(
        row.annotated_path
    )

    reviewed_mask_path = Path(
        row.reviewed_mask
    )

    paths = truth_registration_paths(
        page
    )


    try:

        require(
            clean_path is not None,
            (
                f"{page}: no source clean page "
                f"resolved for identity {clean_key}."
            ),
        )

        require(
            observed_path.exists(),
            (
                f"{page}: frozen observed page missing: "
                f"{observed_path}"
            ),
        )

        require(
            reviewed_mask_path.exists(),
            (
                f"{page}: reviewed mask missing: "
                f"{reviewed_mask_path}"
            ),
        )


        fingerprint = truth_registration_fingerprint(
            clean_path,
            observed_path,
        )


        reuse = False


        if (
            not FORCE_TRUTH_REGISTRATION_REBUILD
            and all(
                path.exists()
                for path
                in paths.values()
            )
        ):

            try:

                old_diagnostics = json.loads(
                    paths[
                        "diagnostics"
                    ].read_text()
                )

                reuse = (
                    old_diagnostics.get(
                        "fingerprint"
                    )
                    == fingerprint
                )

                if reuse:

                    expected_hashes = (
                        old_diagnostics.get(
                            "output_hashes",
                            {},
                        )
                    )

                    actual_hashes = {
                        "registered_sha256":
                            sha256_file(
                                paths[
                                    "registered"
                                ]
                            ),

                        "preview_sha256":
                            sha256_file(
                                paths[
                                    "preview"
                                ]
                            ),

                        "homography_sha256":
                            sha256_file(
                                paths[
                                    "homography"
                                ]
                            ),

                        "validity_sha256":
                            sha256_file(
                                paths[
                                    "validity"
                                ]
                            ),
                    }

                    reuse = (
                        expected_hashes
                        == actual_hashes
                    )

            except Exception:

                reuse = False


        if reuse:

            registered_clean = read_gray(
                paths[
                    "registered"
                ]
            )

            registered_validity = read_gray(
                paths[
                     "validity"
                ]
            )

            homography = np.load(
                paths[
                    "homography"
                ]
            )

            saved = json.loads(
                paths[
                    "diagnostics"
                ].read_text()
            )

            diagnostics = saved[
                "registration"
            ]

            status = "REUSED"


        else:

            clean = read_gray(
                clean_path
            )

            observed = read_gray(
                observed_path
            )

            reviewed_mask = read_gray(
                reviewed_mask_path
           )


            require(
                observed.shape
                == reviewed_mask.shape,
                (
                    f"{page}: frozen observed/reviewed-mask "
                    f"shape mismatch: observed={observed.shape}, "
                    f"mask={reviewed_mask.shape}."
                ),
            )


            (
                registered_clean,
                registered_validity,
                homography,
                diagnostics,
            ) = build_clean_to_observed_registration(
                clean,
                observed,
                page,
            )


            preview = truth_preview(
                observed,
                registered_clean,
                registered_validity,
            )


            require(
                cv2.imwrite(
                    str(
                        paths[
                            "registered"
                        ]
                    ),
                    registered_clean,
                ),
                (
                    f"{page}: failed to write "
                    "registered clean page."
                ),
            )

            require(
                cv2.imwrite(
                    str(
                        paths[
                            "validity"
                        ]
                    ),
                    registered_validity,
                ),
                (
                    f"{page}: failed to write truth "
                    "registration-validity mask."
                ),
            )

            require(
                cv2.imwrite(
                    str(
                        paths[
                            "preview"
                        ]
                    ),
                    preview,
                ),
                (
                    f"{page}: failed to write "
                    "registration preview."
                ),
            )

            np.save(
                paths[
                    "homography"
                ],
                homography,
            )


            output_hashes = {
                "registered_sha256":
                    sha256_file(
                        paths[
                            "registered"
                        ]
                    ),

                "preview_sha256":
                    sha256_file(
                        paths[
                            "preview"
                        ]
                    ),

                "homography_sha256":
                    sha256_file(
                        paths[
                            "homography"
                        ]
                    ),

                "validity_sha256":
                    sha256_file(
                        paths[
                            "validity"
                        ]
                    ),
            }


            record = {
                "created_utc":
                    datetime.now(
                        timezone.utc
                    ).isoformat(),

                "population_role":
                    "PAIRED_HITL_DEVELOPMENT",

                "truth_role":
                    "POST_LOCK_REGISTRATION_AND_SCORING_ONLY",

                "page":
                    page,

                "fingerprint":
                    fingerprint,

                "registration":
                    diagnostics,

                "output_hashes":
                    output_hashes,

                "clean_source":
                    str(
                        clean_path
                    ),

                "observed_target":
                    str(
                        observed_path
                    ),

                "reviewed_mask":
                    str(
                        reviewed_mask_path
                    ),

                "registered_clean":
                    str(
                        paths[
                            "registered"
                        ]
                    ),

                "validity_mask":
                    str(
                        paths[
                            "validity"
                        ]
                    ),

                "preview":
                    str(
                        paths[
                            "preview"
                        ]
                    ),

                "homography":
                    str(
                        paths[
                            "homography"
                        ]
                    ),
            }


            paths[
                "diagnostics"
            ].write_text(
                json.dumps(
                    record,
                    indent=2,
                )
            )

            status = "BUILT"


        observed = read_gray(
            observed_path
        )

        reviewed_mask = read_gray(
            reviewed_mask_path
        )


        require(
            registered_clean.shape
            == observed.shape
            == reviewed_mask.shape
            == registered_validity.shape,
            (
                f"{page}: post-registration shape mismatch: "
                f"truth={registered_clean.shape}, "
                f"observed={observed.shape}, "
                f"mask={reviewed_mask.shape}, "
                f"validity={registered_validity.shape}."
            ),
        )


        # Historical truth used a one-pixel tolerance in the
        # original clean coordinate system.
        #
        # Convert that into the target coordinate system.
        #
        # For the reported 2.5x scale this becomes radius 3.

        target_scale = max(
            float(
                diagnostics[
                    "dimension_scale_x"
                ]
            ),
            float(
                diagnostics[
                    "dimension_scale_y"
                ]
            ),
        )

        truth_tolerance_radius = max(
            1,
            int(
                np.ceil(
                    target_scale
                )
            ),
        )


        registered_clean_map[
            page
        ] = paths[
            "registered"
        ]

        registered_truth_validity_map[
            page
        ] = paths[
            "validity"
        ]

        truth_tolerance_radius_by_page[
            page
        ] = truth_tolerance_radius


        truth_registration_rows.append({
            "page":
                page,

            "status":
                status,

            "clean_source":
                str(
                    clean_path
                ),

            "observed_target":
                str(
                    observed_path
                ),

            "reviewed_mask":
                str(
                    reviewed_mask_path
                ),

            "clean_source_shape":
                "x".join(
                    str(x)
                    for x
                    in diagnostics[
                        "clean_shape"
                    ]
                ),

            "target_observed_shape":
                "x".join(
                    str(x)
                    for x
                    in diagnostics[
                        "observed_shape"
                    ]
                ),

            "dimension_scale_y":
                float(
                    diagnostics[
                        "dimension_scale_y"
                    ]
                ),

            "dimension_scale_x":
                float(
                    diagnostics[
                        "dimension_scale_x"
                    ]
                ),

            "good_matches":
                int(
                    diagnostics[
                        "good_matches"
                    ]
                ),

            "inliers":
                int(
                    diagnostics[
                        "inliers"
                    ]
                ),

            "inlier_fraction":
                float(
                    diagnostics[
                        "inlier_fraction"
                    ]
                ),

            "median_reprojection_error_target_pixels":
                float(
                    diagnostics[
                        "median_reprojection_error_target_pixels"
                    ]
                ),

            "p95_reprojection_error_target_pixels":
                float(
                    diagnostics[
                        "p95_reprojection_error_target_pixels"
                    ]
                ),

            "target_coverage":
                float(
                    diagnostics[
                        "target_coverage"
                    ]
                ),

            "truth_tolerance_radius_target_pixels":
                truth_tolerance_radius,

            "registered_clean":
                str(
                    paths[
                        "registered"
                    ]
                ),

            "registered_clean_sha256":
                sha256_file(
                    paths[
                        "registered"
                    ]
                ),

            "validity_mask":
                str(
                    paths[
                        "validity"
                    ]
                ),

            "validity_mask_sha256":
                sha256_file(
                    paths[
                        "validity"
                    ]
                ),

            "homography":
                str(
                    paths[
                        "homography"
                    ]
                ),

            "homography_sha256":
                sha256_file(
                    paths[
                        "homography"
                    ]
                ),

            "preview":
                str(
                    paths[
                        "preview"
                    ]
                ),

            "diagnostics":
                str(
                    paths[
                        "diagnostics"
                    ]
                ),
        })


        print(
            page,
            status,
            "| scale:",
            (
                f"{diagnostics['dimension_scale_y']:.4f}"
                " x "
                f"{diagnostics['dimension_scale_x']:.4f}"
            ),
            "| matches:",
            diagnostics[
                "good_matches"
            ],
            "| inliers:",
            diagnostics[
                "inliers"
            ],
            "| median error:",
            (
                f"{diagnostics['median_reprojection_error_target_pixels']:.3f}px"
            ),
            "| coverage:",
            (
                f"{100 * diagnostics['target_coverage']:.2f}%"
            ),
            "| truth tolerance:",
            truth_tolerance_radius,
            "px",
        )


    except Exception as error:

        truth_registration_failures.append({
            "page":
                page,

            "clean_source":
                (
                    str(
                        clean_path
                    )
                    if clean_path is not None
                    else ""
                ),

            "observed_target":
                str(
                    observed_path
                ),

            "reviewed_mask":
                str(
                    reviewed_mask_path
                ),

            "error":
                str(
                    error
                ),
        })

        print(
            page,
            "TRUTH REGISTRATION FAILED:",
            error,
        )


# ============================================================
# Write registration evidence before fail-closed decision
# ============================================================

truth_registration_table = pd.DataFrame(
    truth_registration_rows
)


truth_registration_table_path = (
    TRUTH_REGISTRATION_ROOT
    / "truth_registration_manifest.csv"
)


truth_registration_table.to_csv(
    truth_registration_table_path,
    index=False,
)


truth_registration_failures_path = (
    TRUTH_REGISTRATION_ROOT
    / "truth_registration_failures.csv"
)


pd.DataFrame(
    truth_registration_failures,
    columns=[
        "page",
        "clean_source",
        "observed_target",
        "reviewed_mask",
        "error",
    ],
).to_csv(
    truth_registration_failures_path,
    index=False,
)


require(
    not truth_registration_failures,
    (
        f"{len(truth_registration_failures)} truth "
        "registration failure(s). "
        f"See {truth_registration_failures_path}. "
        "Do not continue to truth scoring."
    ),
)


require(
    len(
        truth_registration_table
    )
    == len(
        truth_manifest
    ),
    (
        "Truth-registration manifest does not cover the "
        "complete locked paired-HITL roster."
    ),
)


require(
    not truth_registration_table[
        "page"
    ].duplicated().any(),
    "Duplicate page in truth-registration manifest.",
)


# ============================================================
# Registration lock
# ============================================================

truth_registration_lock = {
    "state":
        "POST_PREDICTION_TRUTH_REGISTRATION_COMPLETE",

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),

    "population_role":
        "PAIRED_HITL_DEVELOPMENT",

    "truth_role":
        "REGISTRATION_AND_POST_HOC_SCORING_ONLY",

    "pages":
        int(
            len(
                truth_registration_table
            )
        ),

    "direction":
        "CLEAN_TO_FROZEN_NOTEBOOK07_OBSERVED",

    "settings":
        TRUTH_REGISTRATION_SETTINGS,

    "settings_sha256":
        TRUTH_REGISTRATION_SETTINGS_SHA256,

    "review_lock_sha256":
        sha256_file(
            LOCK_DIR
            / "paired_hitl_review_lock.json"
        ),

    "prediction_lock_sha256":
        sha256_file(
            RUN
            / "prediction_lock.json"
        ),

    "truth_registration_manifest":
        str(
            truth_registration_table_path
        ),

    "truth_registration_manifest_sha256":
        sha256_file(
            truth_registration_table_path
        ),

    "failures":
        0,
}


truth_registration_lock_path = (
    TRUTH_REGISTRATION_ROOT
    / "truth_registration_lock.json"
)


truth_registration_lock_path.write_text(
    json.dumps(
        truth_registration_lock,
        indent=2,
    )
)


# ============================================================
# Visual inspection contact sheet
#
# Green = registered clean
# Red   = observed
# Yellow/dark agreement indicates registration agreement
# Blue  = outside valid registered-clean coverage
# ============================================================

preview_rows = truth_registration_table.sort_values(
    [
        "median_reprojection_error_target_pixels",
        "page",
    ],
    ascending=[
        False,
        True,
    ],
).reset_index(
    drop=True
)


preview_count = min(
    12,
    len(
        preview_rows
    ),
)


if preview_count:

    figure, axes = plt.subplots(
        preview_count,
        3,
        figsize=(
            15,
            5 * preview_count,
        ),
        squeeze=False,
    )


    for index, (_, row) in enumerate(
        preview_rows.head(
            preview_count
        ).iterrows()
    ):

        observed = read_gray(
            row.observed_target
        )

        registered_clean = read_gray(
            row.registered_clean
        )

        preview = cv2.imread(
            str(
                row.preview
            ),
            cv2.IMREAD_COLOR,
        )

        require(
            preview is not None,
            (
                f"{row.page}: registration preview "
                "is unreadable."
            ),
        )

        preview = cv2.cvtColor(
            preview,
            cv2.COLOR_BGR2RGB,
        )


        axes[
            index,
            0
        ].imshow(
            observed,
            cmap="gray",
            vmin=0,
            vmax=255,
        )

        axes[
            index,
            0
        ].set_title(
            (
                f"{row.page}\n"
                "Frozen Notebook 07 observed"
            )
        )


        axes[
            index,
            1
        ].imshow(
            registered_clean,
            cmap="gray",
            vmin=0,
            vmax=255,
        )

        axes[
            index,
            1
        ].set_title(
            (
                f"{row.page}\n"
                "Registered clean truth"
            )
        )


        axes[
            index,
            2
        ].imshow(
            preview
        )

        axes[
            index,
            2
        ].set_title(
            (
                f"{row.page}\n"
                f"inliers={int(row.inliers)}"
                f" | median={row.median_reprojection_error_target_pixels:.2f}px"
                f" | coverage={100 * row.target_coverage:.2f}%"
            )
        )


        for column in range(
            3
        ):

            axes[
                index,
                column
            ].axis(
                "off"
            )


    plt.tight_layout()

    registration_contact_sheet_path = (
        TRUTH_REGISTRATION_ROOT
        / "truth_registration_contact_sheet.png"
    )

    plt.savefig(
        registration_contact_sheet_path,
        dpi=140,
        bbox_inches="tight",
    )

    plt.show()


# ============================================================
# Objects supplied to Cell 13
#
# registered_clean_map[page]
# registered_truth_validity_map[page]
# truth_tolerance_radius_by_page[page]
# ============================================================

print()
print("=" * 72)
print(
    "POST-LOCK CLEAN-TRUTH REGISTRATION COMPLETE"
)
print("=" * 72)

print(
    "Registered pages:",
    len(
        registered_clean_map
    )
)

print(
    "Manifest:",
    truth_registration_table_path,
)

print(
    "Lock:",
    truth_registration_lock_path,
)

print(
    "Previews:",
    TRUTH_PREVIEW_DIR,
)

print()
print(
    truth_registration_table[
        [
            "page",
            "status",
            "dimension_scale_y",
            "dimension_scale_x",
            "good_matches",
            "inliers",
            "inlier_fraction",
            "median_reprojection_error_target_pixels",
            "target_coverage",
            "truth_tolerance_radius_target_pixels",
        ]
    ].to_string(
        index=False
    )
)

In [57]:
# @title
# ============================================================
# 13. Truth scoring
# ============================================================

def reconcile_reviewed_mask_canvas(
    seed,
    target_shape,
    page,
):
    """
    Permit only a <=1 pixel bottom/right canvas discrepancy.

    Missing canvas is filled with zero mask authority.
    Existing mask pixels are NEVER rescaled or moved.
    """

    seed = np.asarray(
        seed,
        dtype=np.uint8,
    )

    target_h, target_w = target_shape
    seed_h, seed_w = seed.shape

    dh = target_h - seed_h
    dw = target_w - seed_w

    require(
        dh in (0, 1)
        and dw in (0, 1),
        (
            f"{page}: reviewed-mask canvas mismatch is "
            f"too large to reconcile safely: "
            f"seed={seed.shape}, target={target_shape}"
        ),
    )

    if dh == 0 and dw == 0:
        return seed

    fixed = np.zeros(
        target_shape,
        dtype=np.uint8,
    )

    fixed[
        :seed_h,
        :seed_w
    ] = seed

    print(
        f"{page}: reconciled reviewed-mask canvas "
        f"{seed.shape} -> {target_shape}; "
        f"added {dh} bottom row(s), "
        f"{dw} right column(s), all zero-authority."
    )

    return fixed

def printed_ink(clean):
    return cv2.adaptiveThreshold(
        clean, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV,
        41, 15,
    ) > 0

score_rows = []
pixel_parts = []

for _, r in review_manifest.iterrows():
    page = str(r.page)
    d = RUN/'pages'/page
    obs = read_gray(r.annotated_path)

    require(
        page in registered_clean_map,
        f"{page}: no registered clean truth",
    )

    clean = read_gray(
        registered_clean_map[page]
    )

    truth_valid = (
        read_gray(
            registered_truth_validity_map[page]
        )
        > 0
    )

    truth_tolerance_radius = (
        truth_tolerance_radius_by_page[page]
    )

    seed = read_gray(r.reviewed_mask)

    seed = reconcile_reviewed_mask_canvas(
        seed,
        obs.shape,
        page,
    )

    require(
        clean.shape
        == obs.shape
        == seed.shape
        == truth_valid.shape,
        (
            f"{page}: registered truth shape mismatch: "
            f"clean={clean.shape}, "
            f"observed={obs.shape}, "
            f"seed={seed.shape}, "
            f"validity={truth_valid.shape}"
        ),
    )

    require(obs.shape == clean.shape == seed.shape, f'{page}: truth shape mismatch')

    clean_print = printed_ink(clean)

    truth_kernel = cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (
            2 * truth_tolerance_radius + 1,
            2 * truth_tolerance_radius + 1,
        ),
    )

    clean_tol = (
        cv2.dilate(
            clean_print.astype(np.uint8),
            truth_kernel,
        )
        > 0
    )

    # No truth labels outside valid warped clean-page coverage.
    clean_tol &= truth_valid

    try:
        np.load(d/'arrays.npz')
    except:
        continue

    with np.load(d/'arrays.npz') as zz:
        ambiguous = zz['ambiguous'] > 0
        p_print = zz['p_print'].astype(float)
        p_annotation = zz['p_annotation'].astype(float)
        masks = {name: zz[f'mask__{name}'] > 0 for name in CANDIDATES if name != 'A1_BASELINE'}

    truth_print = (
        ambiguous
        & truth_valid
        & clean_tol
    )

    truth_annotation = (
        ambiguous
        & truth_valid
        & (obs < 200)
        & ~clean_tol
    )

    truth_uncertain = (
        ambiguous
        & (
            ~truth_valid
            | (
                truth_valid
                & ~truth_print
                & ~truth_annotation
            )
        )
    )

    for name in CANDIDATES:
        img = read_gray(d/f'{name}.png')
        if name == 'A1_BASELINE':
            cleanup = np.zeros(obs.shape,bool)
        else:
            cleanup = masks[name]
        score_rows.append({
            'page': page,
            'arm': name,
            'ambiguous_pixels': int(ambiguous.sum()),
            'cleanup_pixels': int(cleanup.sum()),
            'truth_annotation_cleaned': int((cleanup & truth_annotation).sum()),
            'truth_print_lost': int((cleanup & truth_print).sum()),
            'truth_uncertain_changed': int((cleanup & truth_uncertain).sum()),
            'annotation_population': int(truth_annotation.sum()),
            'print_population': int(truth_print.sum()),
            'annotation_cleanup_rate': float((cleanup & truth_annotation).sum()/max(truth_annotation.sum(),1)),
            'print_loss_rate': float((cleanup & truth_print).sum()/max(truth_print.sum(),1)),
        })

    ys,xs = np.where(ambiguous)
    if len(ys):
        part = pd.DataFrame({
            'page': page, 'y': ys, 'x': xs,
            'p_annotation': p_annotation[ys,xs],
            'p_print': p_print[ys,xs],
            'truth_print': truth_print[ys,xs],
            'truth_annotation': truth_annotation[ys,xs],
            'truth_uncertain': truth_uncertain[ys,xs],
        })
        for name,mask in masks.items():
            part[f'{name}_selected'] = mask[ys,xs]
        pixel_parts.append(part)

scores = pd.DataFrame(score_rows)
scores.to_csv(RUN/'tables'/'truth_scores_by_page.csv', index=False)
pixels = pd.concat(pixel_parts, ignore_index=True) if pixel_parts else pd.DataFrame()
pixels.to_parquet(RUN/'tables'/'paired_hitl_ambiguous_truth.parquet', index=False)

pooled = scores.groupby('arm',as_index=False).agg(
    pages=('page','nunique'),
    cleanup_pixels=('cleanup_pixels','sum'),
    truth_annotation_cleaned=('truth_annotation_cleaned','sum'),
    truth_print_lost=('truth_print_lost','sum'),
    truth_uncertain_changed=('truth_uncertain_changed','sum'),
    annotation_population=('annotation_population','sum'),
    print_population=('print_population','sum'),
)
pooled['annotation_cleanup_rate'] = pooled.truth_annotation_cleaned / pooled.annotation_population.clip(lower=1)
pooled['print_loss_rate'] = pooled.truth_print_lost / pooled.print_population.clip(lower=1)
pooled['annotation_per_print_lost'] = pooled.truth_annotation_cleaned / pooled.truth_print_lost.clip(lower=1)
pooled.to_csv(RUN/'tables'/'truth_scores_pooled.csv', index=False)
display(pooled)

,arm,pages,cleanup_pixels,truth_annotation_cleaned,truth_print_lost,...,annotation_population,print_population,annotation_cleanup_rate,print_loss_rate,annotation_per_print_lost
0,A1_BASELINE,59,0,0,0,...,119421,191468,0.000000,0.000000,0.000000
1,C085_P010,59,36064,16685,19375,...,119421,191468,0.139716,0.101192,0.861161
2,C0875_P010,59,13881,4987,8894,...,119421,191468,0.041760,0.046452,0.560715
3,C090_P010,59,976,630,346,...,119421,191468,0.005275,0.001807,1.820809


In [58]:
# @title
# ============================================================
# 14. Page-clustered bootstrap
# ============================================================

BOOTSTRAP_RESAMPLES = 10_000
rng = np.random.default_rng(RANDOM_SEED)
bootstrap_rows = []

for arm, group in scores.groupby('arm'):
    if arm == 'A1_BASELINE':
        continue
    rows = group.reset_index(drop=True)
    ann = rows.annotation_cleanup_rate.to_numpy(float)
    prn = rows.print_loss_rate.to_numpy(float)
    ann_means = np.empty(BOOTSTRAP_RESAMPLES)
    prn_means = np.empty(BOOTSTRAP_RESAMPLES)
    for i in range(BOOTSTRAP_RESAMPLES):
        idx = rng.integers(0, len(rows), len(rows))
        ann_means[i] = ann[idx].mean()
        prn_means[i] = prn[idx].mean()
    bootstrap_rows.append({
        'arm': arm,
        'pages': len(rows),
        'mean_annotation_cleanup_rate': float(ann.mean()),
        'annotation_ci95_low': float(np.quantile(ann_means,0.025)),
        'annotation_ci95_high': float(np.quantile(ann_means,0.975)),
        'mean_print_loss_rate': float(prn.mean()),
        'print_loss_ci95_low': float(np.quantile(prn_means,0.025)),
        'print_loss_ci95_high': float(np.quantile(prn_means,0.975)),
        'pages_with_print_loss': int((rows.truth_print_lost>0).sum()),
    })

bootstrap = pd.DataFrame(bootstrap_rows)
bootstrap.to_csv(RUN/'tables'/'page_clustered_bootstrap.csv', index=False)
display(bootstrap)

,arm,pages,mean_annotation_cleanup_rate,annotation_ci95_low,annotation_ci95_high,mean_print_loss_rate,print_loss_ci95_low,print_loss_ci95_high,pages_with_print_loss
0,C085_P010,59,0.139875,0.099512,0.185420,0.071811,0.038314,0.110919,25
1,C0875_P010,59,0.046564,0.022481,0.078035,0.037987,0.014238,0.066950,15
2,C090_P010,59,0.017107,0.000548,0.044901,0.002301,0.000000,0.006225,3


In [59]:
# @title
# ============================================================
# 15. Draft-to-reviewed morphology and reviewer intervention
# ============================================================

review_change_rows = []
for _, r in review_manifest.iterrows():
    page = str(r.page)
    draft = read_gray(r.draft_mask) > 0
    reviewed = read_gray(r.reviewed_mask) > 0
    added = reviewed & ~draft
    erased = draft & ~reviewed
    retained = reviewed & draft
    review_change_rows.append({
        'page': page,
        'draft_pixels': int(draft.sum()),
        'reviewed_pixels': int(reviewed.sum()),
        'added_pixels': int(added.sum()),
        'erased_pixels': int(erased.sum()),
        'retained_draft_pixels': int(retained.sum()),
        'reviewed_to_draft_ratio': float(reviewed.sum()/max(draft.sum(),1)),
        'added_fraction_of_reviewed': float(added.sum()/max(reviewed.sum(),1)),
        'erased_fraction_of_draft': float(erased.sum()/max(draft.sum(),1)),
    })
review_changes = pd.DataFrame(review_change_rows)
review_changes.to_csv(RUN/'tables'/'draft_to_reviewed_changes.csv', index=False)
display(review_changes.describe())

,draft_pixels,reviewed_pixels,added_pixels,erased_pixels,retained_draft_pixels,reviewed_to_draft_ratio,added_fraction_of_reviewed,erased_fraction_of_draft
count,83.000000,83.000000,83.000000,83.000000,83.000000,83.000000,83.000000,83.000000
mean,80377.012048,74979.542169,2893.867470,8291.337349,72085.674699,0.704142,0.059215,0.084692
std,97364.791475,88989.378776,5988.673915,14556.028235,87868.235009,0.523018,0.168470,0.169741
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,49944.000000,49944.000000,0.000000,900.000000,48318.000000,0.920304,0.000000,0.015146
75%,130964.500000,126429.000000,3066.500000,9757.000000,115867.500000,1.000000,0.038800,0.102294
max,421987.000000,401208.000000,35816.000000,60994.000000,400862.000000,2.480270,1.000000,1.000000


In [ ]:
# @title
# ============================================================
# 16. Truth atlas and visual summaries
# ============================================================

VIS = RUN/'visuals'
VIS.mkdir(exist_ok=True)

# Frontier
fig, ax = plt.subplots(figsize=(9,7))
for _, row in bootstrap.iterrows():
    ax.scatter(row.mean_print_loss_rate*100, row.mean_annotation_cleanup_rate*100, s=90)
    ax.text(row.mean_print_loss_rate*100, row.mean_annotation_cleanup_rate*100, ' '+row.arm, va='center')
ax.set_xlabel('Mean genuine-print loss (%)')
ax.set_ylabel('Mean additional annotation cleanup (%)')
ax.set_title('Production-profile paired HITL A1 cleanup frontier')
ax.grid(True, alpha=.3)
fig.tight_layout()
fig.savefig(VIS/'paired_hitl_cleanup_frontier.png', dpi=180)
plt.close(fig)

# Rank pages by print loss, then cleanup opportunity.
rank = scores[scores.arm!='A1_BASELINE'].copy()
rank['risk_score'] = rank.truth_print_lost*1000 + rank.truth_annotation_cleaned
rank = rank.sort_values('risk_score', ascending=False)
rank.to_csv(RUN/'tables'/'truth_atlas_review_queue.csv', index=False)

# Create one contact sheet per top 24 page/arm rows.
def rgb(gray): return cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)

def overlay_mask(gray, mask, color, alpha=.55):
    b = rgb(gray); o=b.copy(); m=np.asarray(mask,bool); c=np.array(color,np.float32)
    o[m]=np.clip((1-alpha)*b[m]+alpha*c,0,255).astype(np.uint8); return o

for rank_index, row in enumerate(rank.head(24).itertuples(), start=1):
    page = str(row.page); arm=str(row.arm); d=RUN/'pages'/page

    # ------------------------------------------------------------
    # All arrays used in the truth atlas MUST be in the frozen
    # Notebook-07 observed coordinate system.
    # ------------------------------------------------------------

    obs = read_gray(
        review_manifest
        .set_index("page")
        .loc[
            page,
            "annotated_path",
        ]
    )

    require(
        page in registered_clean_map,
        f"{page}: registered clean truth missing",
    )

    require(
        page in registered_truth_validity_map,
        f"{page}: registered truth-validity mask missing",
    )

    require(
        page in truth_tolerance_radius_by_page,
        f"{page}: truth-tolerance radius missing",
    )


    # IMPORTANT:
    # use Cell 12A's REGISTERED clean truth,
    # NOT clean_map[page].
    clean = read_gray(
        registered_clean_map[
            page
        ]
    )

    truth_valid = (
        read_gray(
            registered_truth_validity_map[
                page
            ]
        )
        > 0
    )

    truth_tolerance_radius = int(
        truth_tolerance_radius_by_page[
            page
        ]
    )


    a1 = read_gray(
        d / "A1_BASELINE.png"
    )

    cand = read_gray(
        d / f"{arm}.png"
    )


    with np.load(
        d / "arrays.npz"
    ) as zz:

        p_ann = (
            zz["p_annotation"]
            .astype(float)
        )

        p_pr = (
            zz["p_print"]
            .astype(float)
        )

        selection = (
            zz[
                f"mask__{arm}"
            ]
            > 0
        )


    # ------------------------------------------------------------
    # Fail BEFORE any pixelwise truth operation if coordinate
    # identity has been lost.
    # ------------------------------------------------------------

    expected_shape = (
        selection.shape
    )

    for name, array in {
        "observed":
            obs,

        "registered_clean":
            clean,

        "truth_valid":
            truth_valid,

        "A1":
            a1,

        "candidate":
            cand,

        "p_annotation":
            p_ann,

        "p_print":
            p_pr,
    }.items():

        require(
            array.shape
            == expected_shape,
            (
                f"{page}/{arm}: "
                f"{name} shape {array.shape} "
                f"!= selection shape {expected_shape}"
            ),
        )

    with np.load(d/'arrays.npz') as zz:
        p_ann=zz['p_annotation'].astype(float); p_pr=zz['p_print'].astype(float); selection=zz[f'mask__{arm}']>0
    clean_print=printed_ink(clean);
    truth_kernel=cv2.getStructuringElement(
        cv2.MORPH_ELLIPSE,
        (
            2 * truth_tolerance_radius + 1,
            2 * truth_tolerance_radius + 1,
        ),
    )
    clean_tol=cv2.dilate(clean_print.astype(np.uint8),truth_kernel)>0
    clean_tol &= truth_valid
    print_loss=selection & truth_valid & clean_tol
    annotation_gain=selection & truth_valid & (obs<200) & ~clean_tol
    fig,axes=plt.subplots(2,4,figsize=(16,11))
    panels=[
        (rgb(obs),'Annotated'),(rgb(clean),'Clean truth'),(rgb(a1),'A1'),(rgb(cand),arm),
        (overlay_mask(obs,selection,(255,255,0)),'Selected'),
        (overlay_mask(obs,annotation_gain,(0,255,0)),'Truth annotation cleaned'),
        (overlay_mask(obs,print_loss,(255,0,0)),'Truth print lost'),
        (cv2.cvtColor(cv2.applyColorMap(np.clip(p_ann*255,0,255).astype(np.uint8),cv2.COLORMAP_TURBO),cv2.COLOR_BGR2RGB),'p(annotation)'),
    ]
    for ax,(im,title) in zip(axes.flat,panels): ax.imshow(im,cmap='gray');ax.set_title(title);ax.axis('off')
    fig.suptitle(f'{rank_index}. {page} / {arm} — annotation cleaned {row.truth_annotation_cleaned:,}; print lost {row.truth_print_lost:,}')
    fig.tight_layout()
    fig.savefig(VIS/f'truth_atlas_{rank_index:02d}_{page}_{arm}.png',dpi=140)
    plt.close(fig)

print('Visuals:', VIS)

ValueError: operands could not be broadcast together with shapes (2363,1670) (3507,2480) (2363,1670) 

In [61]:
# @title
# ============================================================
# 17. Final descriptive verdict and bundle
# ============================================================

best = None
if len(bootstrap):
    # Descriptive only: lowest upper print-loss bound, then greatest cleanup.
    best = bootstrap.sort_values(['print_loss_ci95_high','mean_annotation_cleanup_rate'], ascending=[True,False]).iloc[0].to_dict()

verdict = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'notebook': '87_Paired_HITL_Development_Truth_Transport',
    'population_role': 'PAIRED_HITL_DEVELOPMENT',
    'review_lock_state': 'CLOSED_BEFORE_CLEAN_TRUTH',
    'prediction_lock_state': 'CLOSED_BEFORE_CLEAN_TRUTH',
    'pages': int(len(review_manifest)),
    'candidates': CANDIDATES,
    'descriptive_best_by_lowest_upper_print_loss': best,
    'automatic_promotion': False,
    'independent_validation': False,
    'verdict': 'DIAGNOSTIC_COMPLETE_NO_AUTOMATIC_WINNER',
}
(RUN/'notebook87_verdict.json').write_text(json.dumps(verdict, indent=2))

summary = [
    'NOTEBOOK 87 — PAIRED HITL DEVELOPMENT TRUTH TRANSPORT',
    '',
    f'Pages reviewed and scored: {len(review_manifest)}',
    'Population: PAIRED_HITL_DEVELOPMENT',
    'Clean truth was unavailable during review and candidate generation.',
    '',
    'Candidates:',
] + [f'  {k}: {v}' for k,v in CANDIDATES.items()] + [
    '',
    'No automatic promotion is authorised.',
    'Existing paired pages are development-exposed and cannot provide final independent promotion evidence.',
]
(RUN/'executive_summary.txt').write_text('\n'.join(summary))

upload = RUN/'upload'
for p in [
    RUN/'executive_summary.txt', RUN/'notebook87_verdict.json',
    RUN/'tables'/'truth_scores_pooled.csv', RUN/'tables'/'page_clustered_bootstrap.csv',
    RUN/'tables'/'draft_to_reviewed_changes.csv', RUN/'tables'/'truth_atlas_review_queue.csv',
    VIS/'paired_hitl_cleanup_frontier.png', LOCK_DIR/'paired_hitl_review_lock.json',
    RUN/'prediction_lock.json',
]:
    require(p.exists(), f'Missing bundle artefact: {p}')
    shutil.copy2(p, upload/p.name)

manifest=[]
for p in sorted(upload.iterdir()):
    if p.is_file(): manifest.append({'file':p.name,'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(manifest).to_csv(upload/'upload_manifest.csv',index=False)
zip_base=RUN/'notebook87_paired_hitl_truth_transport_bundle'
shutil.make_archive(str(zip_base),'zip',upload)
print('Bundle:',str(zip_base)+'.zip')
print(json.dumps(verdict,indent=2))

Bundle: /content/drive/MyDrive/MusicAnnotationProject/paired_hitl_development/runs/20261121_164815_notebook87/notebook87_paired_hitl_truth_transport_bundle.zip
{
  "created_utc": "2026-11-21T21:18:11.475837+00:00",
  "notebook": "87_Paired_HITL_Development_Truth_Transport",
  "population_role": "PAIRED_HITL_DEVELOPMENT",
  "review_lock_state": "CLOSED_BEFORE_CLEAN_TRUTH",
  "prediction_lock_state": "CLOSED_BEFORE_CLEAN_TRUTH",
  "pages": 83,
  "candidates": {
    "A1_BASELINE": null,
    "C090_P010": [
      0.9,
      0.1
    ],
    "C0875_P010": [
      0.875,
      0.1
    ],
    "C085_P010": [
      0.85,
      0.1
    ]
  },
  "descriptive_best_by_lowest_upper_print_loss": {
    "arm": "C090_P010",
    "pages": 59,
    "mean_annotation_cleanup_rate": 0.01710683099443034,
    "annotation_ci95_low": 0.0005475394070429012,
    "annotation_ci95_high": 0.04490067034638153,
    "mean_print_loss_rate": 0.0023013451108396953,
    "print_loss_ci95_low": 0.0,
    "print_loss_ci95_high": 0.0